# Phase 3 — Nettoyage des données NBA

## Objectif

Appliquer les stratégies de nettoyage appropriées aux problèmes identifiés
pendant la Phase 2, tout en documentant et justifiant chaque transformation.

Le nettoyage suit strictement l'ordre recommandé par le guide :

1. Doublons
2. Erreurs évidentes
3. Types de données
4. Standardisation
5. Valeurs manquantes
6. Outliers
7. Validation finale

> Les données originales ne sont jamais modifiées.
> Toutes les transformations sont appliquées sur des copies.

### Règle contre le data leakage

Aucune imputation par moyenne, médiane ou mode, aucune winsorisation,
aucune interpolation et aucune transformation statistique apprise ne sera
effectuée pendant cette phase.

Ces traitements devront être réalisés dans la pipeline ML après le split
train/test.

# Etape 1 — Préparation

In [4]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_colwidth", 120)



# CHEMINS DU PROJET


PROJECT_ROOT = Path.cwd()

if (
    not (PROJECT_ROOT / "data").exists()
    and (PROJECT_ROOT / "ML_NBA_Project" / "data").exists()
):
    PROJECT_ROOT = PROJECT_ROOT / "ML_NBA_Project"


INTERIM_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
)

CLEAN_DIR = (
    PROJECT_ROOT
    / "data"
    / "clean"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "phase3_nettoyage"
)


CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Projet :",
    PROJECT_ROOT.resolve()
)

print(
    "Données nettoyées :",
    CLEAN_DIR.resolve()
)

print(
    "Rapports :",
    REPORT_DIR.resolve()
)


# CHARGEMENT DES SOURCES ORIGINALES


SOURCE_FILES = {
    "player_bio":
        INTERIM_DIR
        / "player_bio_all_seasons.parquet",

    "player_stats_base":
        INTERIM_DIR
        / "player_stats_base_all_seasons.parquet",

    "player_stats_advanced":
        INTERIM_DIR
        / "player_stats_advanced_all_seasons.parquet",

    "team_stats_base":
        INTERIM_DIR
        / "team_stats_base_all_seasons.parquet",

    "team_stats_advanced":
        INTERIM_DIR
        / "team_stats_advanced_all_seasons.parquet",

    "rosters":
        INTERIM_DIR
        / "rosters_all_seasons.parquet",
}


sources_raw = {}


for source_name, path in SOURCE_FILES.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Fichier manquant : {path}"
        )

    sources_raw[source_name] = (
        pd.read_parquet(path)
    )

    print(
        f"{source_name:25s}",
        sources_raw[source_name].shape
    )


# Dataset Kaggle brut
kaggle_candidates = [
    PROJECT_ROOT
    / "data"
    / "raw"
    / "kaggle"
    / "all_seasons.csv",

    PROJECT_ROOT
    / "all_seasons.csv",

    PROJECT_ROOT
    / "all_seasons(2).csv",
]


kaggle_path = next(
    (
        path
        for path in kaggle_candidates
        if path.exists()
    ),
    None
)


if kaggle_path is None:
    raise FileNotFoundError(
        "Dataset Kaggle all_seasons.csv introuvable."
    )


sources_raw[
    "kaggle_all_seasons"
] = pd.read_csv(
    kaggle_path,
    low_memory=False
)


print(
    f"{'kaggle_all_seasons':25s}",
    sources_raw[
        "kaggle_all_seasons"
    ].shape
)


# COPIES DE TRAVAIL


sources_clean = {
    source_name:
        df.copy(deep=True)

    for source_name, df
    in sources_raw.items()
}


print(
    "Sources originales :",
    len(sources_raw)
)

print(
    "Copies de travail :",
    len(sources_clean)
)


for source_name in sources_raw:

    assert (
        sources_raw[source_name]
        is not sources_clean[source_name]
    )


print(
    " Toutes les sources de travail "
    "sont des copies indépendantes."
)


# ÉTAT INITIAL AVANT NETTOYAGE


TEXT_MISSING_VALUES = {
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "<na>",
}


def text_missing_mask(series):

    if not (
        pd.api.types.is_object_dtype(series)
        or pd.api.types.is_string_dtype(series)
    ):
        return pd.Series(
            False,
            index=series.index
        )

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )

    return normalized.isin(
        TEXT_MISSING_VALUES
    )


def count_missing(df):

    total = 0

    for col in df.columns:

        native_missing = (
            df[col].isna()
        )

        textual_missing = (
            text_missing_mask(df[col])
        )

        total += int(
            (
                native_missing
                | textual_missing
            ).sum()
        )

    return total


before_rows = []


for source_name, df in sources_raw.items():

    exact_duplicates = int(
        df.duplicated(
            keep=False
        ).sum()
    )

    before_rows.append({
        "source":
            source_name,

        "lignes":
            len(df),

        "colonnes":
            df.shape[1],

        "cellules_manquantes":
            count_missing(df),

        "lignes_doublons_exacts":
            exact_duplicates,
    })


before_snapshot = pd.DataFrame(
    before_rows
)


before_snapshot.to_csv(
    REPORT_DIR
    / "01_etat_initial.csv",

    index=False,
    encoding="utf-8"
)


display(before_snapshot)

Projet : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model
Données nettoyées : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean
Rapports : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\reports\phase3_nettoyage
player_bio                (12846, 25)
player_stats_base         (12846, 69)
player_stats_advanced     (12846, 81)
team_stats_base           (802, 56)
team_stats_advanced       (802, 48)
rosters                   (11672, 18)
kaggle_all_seasons        (12844, 22)
Sources originales : 7
Copies de travail : 7
 Toutes les sources de travail sont des copies indépendantes.


,source,lignes,colonnes,cellules_manquantes,lignes_doublons_exacts
0,player_bio,12846,25,2233,0
1,player_stats_base,12846,69,0,0
2,player_stats_advanced,12846,81,0,0
3,team_stats_base,802,56,0,0
4,team_stats_advanced,802,48,0,0
5,rosters,11672,18,17167,0
6,kaggle_all_seasons,12844,22,1859,0


In [5]:

# LOG DES TRANSFORMATIONS


cleaning_log_rows = []


def add_cleaning_log(
    source,
    colonne,
    probleme,
    action,
    lignes_affectees,
    justification
):

    cleaning_log_rows.append({
        "#":
            len(cleaning_log_rows) + 1,

        "source":
            source,

        "colonne":
            colonne,

        "probleme":
            probleme,

        "action":
            action,

        "lignes_affectees":
            int(lignes_affectees),

        "justification":
            justification,
    })


def get_cleaning_log():

    return pd.DataFrame(
        cleaning_log_rows,
        columns=[
            "#",
            "source",
            "colonne",
            "probleme",
            "action",
            "lignes_affectees",
            "justification",
        ]
    )


display(
    get_cleaning_log()
)


# EMPREINTE DES DONNÉES ORIGINALES


raw_fingerprints = {}


for source_name, df in sources_raw.items():

    raw_fingerprints[
        source_name
    ] = int(
        pd.util.hash_pandas_object(
            df,
            index=True
        ).sum()
    )


print(
    " Empreintes des sources originales enregistrées."
)

,#,source,colonne,probleme,action,lignes_affectees,justification


 Empreintes des sources originales enregistrées.


## 2. Traitement des doublons

Cette étape traite les doublons conformément au diagnostic réalisé en Phase 2.

Deux situations sont distinguées :

- les doublons exacts, pour lesquels les occurrences supplémentaires peuvent
  être supprimées ;
- les doublons partiels, pour lesquels une investigation est nécessaire avant
  toute suppression.

Les données sont modifiées uniquement dans `sources_clean`.
Les DataFrames originaux stockés dans `sources_raw` restent inchangés.

In [6]:
# Clés métier utilisées pour contrôler l'unicité

SOURCE_KEYS = {
    "player_bio": ["PLAYER_ID", "SEASON"],
    "player_stats_base": ["PLAYER_ID", "SEASON"],
    "player_stats_advanced": ["PLAYER_ID", "SEASON"],
    "team_stats_base": ["TEAM_ID", "SEASON"],
    "team_stats_advanced": ["TEAM_ID", "SEASON"],
    "rosters": ["PLAYER_ID", "TeamID", "SEASON"],
    "kaggle_all_seasons": ["player_name", "season"],
}


# Statistiques avant traitement des doublons

stats_before_duplicates = {}

for source_name, df in sources_clean.items():

    numeric_cols = df.select_dtypes(
        include=[np.number]
    ).columns

    stats_before_duplicates[source_name] = (
        df[numeric_cols]
        .agg(["mean", "median"])
        .T
    )


# Détection et suppression des doublons exacts

duplicate_report_rows = []

for source_name, df in sources_clean.items():

    rows_before = len(df)

    exact_duplicates = int(
        df.duplicated(
            keep="first"
        ).sum()
    )

    sources_clean[source_name] = (
        df.drop_duplicates(
            keep="first"
        ).copy()
    )

    rows_after = len(
        sources_clean[source_name]
    )

    duplicate_report_rows.append({
        "source": source_name,
        "lignes_avant": rows_before,
        "doublons_exacts_supprimes": exact_duplicates,
        "lignes_apres": rows_after,
        "variation_lignes": rows_after - rows_before,
    })

    already_logged = any(
        row["source"] == source_name
        and row["probleme"] == "Doublons exacts"
        for row in cleaning_log_rows
    )

    if not already_logged:

        add_cleaning_log(
            source=source_name,
            colonne="-",
            probleme="Doublons exacts",
            action=(
                "Suppression des occurrences supplémentaires"
                if exact_duplicates > 0
                else "Aucune suppression nécessaire"
            ),
            lignes_affectees=exact_duplicates,
            justification=(
                "Les doublons exacts n'apportent aucune information "
                "supplémentaire. La première occurrence est conservée."
            )
        )


duplicate_report = pd.DataFrame(
    duplicate_report_rows
)

duplicate_report.to_csv(
    REPORT_DIR / "02_doublons_exacts.csv",
    index=False,
    encoding="utf-8"
)

display(duplicate_report)

,source,lignes_avant,doublons_exacts_supprimes,lignes_apres,variation_lignes
0,player_bio,12846,0,12846,0
1,player_stats_base,12846,0,12846,0
2,player_stats_advanced,12846,0,12846,0
3,team_stats_base,802,0,802,0
4,team_stats_advanced,802,0,802,0
5,rosters,11672,0,11672,0
6,kaggle_all_seasons,12844,0,12844,0


In [7]:
# Vérification après suppression des doublons exacts

remaining_exact_duplicates = []

for source_name, df in sources_clean.items():

    remaining_exact_duplicates.append({
        "source": source_name,
        "doublons_exacts_restants": int(
            df.duplicated(
                keep=False
            ).sum()
        )
    })


remaining_exact_duplicates = pd.DataFrame(
    remaining_exact_duplicates
)

display(remaining_exact_duplicates)


assert (
    remaining_exact_duplicates[
        "doublons_exacts_restants"
    ].sum()
    == 0
)

print("Aucun doublon exact restant.")


# Contrôle des doublons sur les clés métier

partial_duplicate_rows = []

for source_name, df in sources_clean.items():

    keys = SOURCE_KEYS[source_name]

    duplicate_mask = df.duplicated(
        subset=keys,
        keep=False
    )

    if not duplicate_mask.any():
        continue

    groups = (
        df.loc[
            duplicate_mask,
            keys
        ]
        .value_counts()
        .reset_index(
            name="nombre_lignes"
        )
    )

    groups.insert(
        0,
        "source",
        source_name
    )

    partial_duplicate_rows.append(
        groups
    )


if partial_duplicate_rows:

    partial_duplicates = pd.concat(
        partial_duplicate_rows,
        ignore_index=True,
        sort=False
    )

else:

    partial_duplicates = pd.DataFrame()


partial_duplicates.to_csv(
    REPORT_DIR / "02_doublons_partiels.csv",
    index=False,
    encoding="utf-8"
)

display(partial_duplicates)

,source,doublons_exacts_restants
0,player_bio,0
1,player_stats_base,0
2,player_stats_advanced,0
3,team_stats_base,0
4,team_stats_advanced,0
5,rosters,0
6,kaggle_all_seasons,0


Aucun doublon exact restant.


,source,player_name,season,nombre_lignes
0,kaggle_all_seasons,Chris Johnson,2012-13,2
1,kaggle_all_seasons,Marcus Williams,2007-08,2
2,kaggle_all_seasons,Marcus Williams,2008-09,2
3,kaggle_all_seasons,Tony Mitchell,2013-14,2


In [8]:
# Recherche des joueurs présents dans plusieurs équipes sur une même saison

rosters = sources_clean["rosters"]

roster_multi_team = (
    rosters
    .groupby(
        ["PLAYER_ID", "SEASON"]
    )["TeamID"]
    .nunique()
    .reset_index(
        name="nombre_equipes"
    )
)

roster_multi_team = roster_multi_team[
    roster_multi_team["nombre_equipes"] > 1
]

display(roster_multi_team)

,PLAYER_ID,SEASON,nombre_equipes
9348,203915,2015-16,2


In [9]:
# Comparaison des statistiques avant et après traitement

stats_impact_rows = []

for source_name, df in sources_clean.items():

    numeric_cols = df.select_dtypes(
        include=[np.number]
    ).columns

    stats_after = (
        df[numeric_cols]
        .agg(["mean", "median"])
        .T
    )

    stats_before = stats_before_duplicates[
        source_name
    ]

    common_cols = stats_before.index.intersection(
        stats_after.index
    )

    comparison = pd.DataFrame({
        "source": source_name,
        "colonne": common_cols,
        "moyenne_avant": stats_before.loc[
            common_cols,
            "mean"
        ].values,
        "moyenne_apres": stats_after.loc[
            common_cols,
            "mean"
        ].values,
        "mediane_avant": stats_before.loc[
            common_cols,
            "median"
        ].values,
        "mediane_apres": stats_after.loc[
            common_cols,
            "median"
        ].values,
    })

    comparison["variation_moyenne"] = (
        comparison["moyenne_apres"]
        - comparison["moyenne_avant"]
    )

    comparison["variation_mediane"] = (
        comparison["mediane_apres"]
        - comparison["mediane_avant"]
    )

    stats_impact_rows.append(
        comparison
    )


duplicate_stats_impact = pd.concat(
    stats_impact_rows,
    ignore_index=True
)


duplicate_stats_impact.to_csv(
    REPORT_DIR / "02_impact_doublons_statistiques.csv",
    index=False,
    encoding="utf-8"
)


impact_summary = (
    duplicate_stats_impact
    .groupby("source")
    .agg(
        variation_max_moyenne=(
            "variation_moyenne",
            lambda x: x.abs().max()
        ),
        variation_max_mediane=(
            "variation_mediane",
            lambda x: x.abs().max()
        ),
    )
    .reset_index()
)


display(impact_summary)

,source,variation_max_moyenne,variation_max_mediane
0,kaggle_all_seasons,0.0,0.0
1,player_bio,0.0,0.0
2,player_stats_advanced,0.0,0.0
3,player_stats_base,0.0,0.0
4,rosters,0.0,0.0
5,team_stats_advanced,0.0,0.0
6,team_stats_base,0.0,0.0


In [10]:
cleaning_log = get_cleaning_log()

cleaning_log.to_csv(
    REPORT_DIR / "cleaning_log.csv",
    index=False,
    encoding="utf-8"
)

display(cleaning_log)

,#,source,colonne,probleme,action,lignes_affectees,justification
0,1,player_bio,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
1,2,player_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
2,3,player_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
3,4,team_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
4,5,team_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
5,6,rosters,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
6,7,kaggle_all_seasons,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.


### Résultat du traitement des doublons

Aucun doublon exact n'a été détecté dans les sept sources. Aucune ligne n'a
donc été supprimée et les statistiques descriptives restent inchangées.

Le contrôle des clés métier met en évidence quatre groupes dans le dataset
Kaggle pour la clé candidate `player_name + season`. Ces groupes correspondent
à des homonymes et représentent des joueurs distincts. Ils sont donc conservés.

La clé `player_name + season` n'est pas suffisamment fiable pour identifier
un joueur. Le dataset final utilisera `PLAYER_ID` comme identifiant joueur.

Dans les rosters, un joueur (`PLAYER_ID = 203915`) est associé à deux équipes
pendant la saison 2015-16. Ce cas est valide et est conservé. La clé du roster
reste `PLAYER_ID + TeamID + SEASON`.

### Checklist — Doublons

- [x] Identifier et supprimer les doublons exacts
- [x] Documenter le nombre de doublons supprimés
- [x] Vérifier l'impact sur les statistiques

### Bilan

- Doublons exacts supprimés : **0**
- Groupes Kaggle à clé candidate non unique : **4**
- Lignes Kaggle concernées : **8**
- Cas joueur-saison multi-équipe dans les rosters : **1**
- Impact sur les statistiques : **aucun**

## 3. Correction des erreurs évidentes

Cette étape vérifie les valeurs manifestement impossibles identifiées lors
du diagnostic de qualité.

Une valeur extrême n'est pas automatiquement considérée comme une erreur.
Elle n'est corrigée ou supprimée que si son caractère erroné est certain.

Les contrôles portent notamment sur :

- les âges impossibles ;
- les nombres de matchs impossibles ;
- les statistiques négatives impossibles ;
- les relations logiques entre statistiques ;
- les pourcentages ayant des bornes métier strictes ;
- les dates impossibles ;
- les catégories manifestement invalides ;
- les colonnes techniques sans valeur métier.

Les données originales de `sources_raw` restent inchangées.

In [11]:
# Contrôles des erreurs évidentes

step3_rows_before = {
    source: len(df)
    for source, df in sources_clean.items()
}

step3_cols_before = {
    source: df.shape[1]
    for source, df in sources_clean.items()
}

error_checks = []


def add_error_check(source, colonne, regle, mask, df):
    mask = pd.Series(mask, index=df.index).fillna(False)

    exemples = ""

    if colonne in df.columns and mask.any():
        exemples = " | ".join(
            df.loc[mask, colonne]
            .astype(str)
            .drop_duplicates()
            .head(10)
            .tolist()
        )

    error_checks.append({
        "source": source,
        "colonne": colonne,
        "regle": regle,
        "nombre_erreurs": int(mask.sum()),
        "exemples": exemples,
    })


# Ages

age_columns = {
    "player_bio": "AGE",
    "player_stats_base": "AGE",
    "player_stats_advanced": "AGE",
    "rosters": "AGE",
    "kaggle_all_seasons": "age",
}

for source_name, col in age_columns.items():
    df = sources_clean[source_name]

    if col not in df.columns:
        continue

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    mask = (
        values.notna()
        & (
            (values < 18)
            | (values > 50)
        )
    )

    add_error_check(
        source_name,
        col,
        "Age compris entre 18 et 50 ans",
        mask,
        df
    )


# Nombre de matchs joueurs

player_gp_columns = {
    "player_bio": "GP",
    "player_stats_base": "GP",
    "player_stats_advanced": "GP",
    "kaggle_all_seasons": "gp",
}

for source_name, col in player_gp_columns.items():
    df = sources_clean[source_name]

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    mask = (
        values.notna()
        & (
            (values < 0)
            | (values > 90)
        )
    )

    add_error_check(
        source_name,
        col,
        "Nombre de matchs joueur compris entre 0 et 90",
        mask,
        df
    )


# Nombre de matchs équipes

for source_name in [
    "team_stats_base",
    "team_stats_advanced",
]:
    df = sources_clean[source_name]

    values = pd.to_numeric(
        df["GP"],
        errors="coerce"
    )

    mask = (
        values.notna()
        & (
            (values < 0)
            | (values > 82)
        )
    )

    add_error_check(
        source_name,
        "GP",
        "Nombre de matchs équipe compris entre 0 et 82",
        mask,
        df
    )

In [12]:
# Valeurs statistiques qui ne peuvent pas être négatives

non_negative_columns = {
    "player_bio": [
        "PLAYER_HEIGHT_INCHES",
        "GP",
        "PTS",
        "REB",
        "AST",
    ],

    "player_stats_base": [
        "GP",
        "W",
        "L",
        "MIN",
        "FGM",
        "FGA",
        "FG3M",
        "FG3A",
        "FTM",
        "FTA",
        "OREB",
        "DREB",
        "REB",
        "AST",
        "TOV",
        "STL",
        "BLK",
        "BLKA",
        "PF",
        "PFD",
        "PTS",
    ],

    "player_stats_advanced": [
        "GP",
        "W",
        "L",
        "MIN",
    ],

    "team_stats_base": [
        "GP",
        "W",
        "L",
        "MIN",
        "FGM",
        "FGA",
        "FG3M",
        "FG3A",
        "FTM",
        "FTA",
        "OREB",
        "DREB",
        "REB",
        "AST",
        "TOV",
        "STL",
        "BLK",
        "BLKA",
        "PF",
        "PFD",
        "PTS",
    ],

    "team_stats_advanced": [
        "GP",
        "W",
        "L",
        "MIN",
    ],

    "kaggle_all_seasons": [
        "player_height",
        "player_weight",
        "gp",
        "pts",
        "reb",
        "ast",
    ],
}


for source_name, columns in non_negative_columns.items():
    df = sources_clean[source_name]

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        mask = (
            values.notna()
            & (values < 0)
        )

        add_error_check(
            source_name,
            col,
            "Valeur supérieure ou égale à 0",
            mask,
            df
        )

In [13]:
# Cohérence entre tirs réussis et tentés

for source_name in [
    "player_stats_base",
    "team_stats_base",
]:
    df = sources_clean[source_name]

    for made, attempted in [
        ("FGM", "FGA"),
        ("FG3M", "FG3A"),
        ("FTM", "FTA"),
    ]:
        if not {made, attempted}.issubset(df.columns):
            continue

        made_values = pd.to_numeric(
            df[made],
            errors="coerce"
        )

        attempted_values = pd.to_numeric(
            df[attempted],
            errors="coerce"
        )

        mask = (
            made_values.notna()
            & attempted_values.notna()
            & (made_values > attempted_values)
        )

        error_checks.append({
            "source": source_name,
            "colonne": f"{made}/{attempted}",
            "regle": f"{made} <= {attempted}",
            "nombre_erreurs": int(mask.sum()),
            "exemples": "",
        })


# W + L = GP

for source_name in [
    "player_stats_base",
    "player_stats_advanced",
    "team_stats_base",
    "team_stats_advanced",
]:
    df = sources_clean[source_name]

    if not {"GP", "W", "L"}.issubset(df.columns):
        continue

    gp = pd.to_numeric(
        df["GP"],
        errors="coerce"
    )

    wins = pd.to_numeric(
        df["W"],
        errors="coerce"
    )

    losses = pd.to_numeric(
        df["L"],
        errors="coerce"
    )

    mask = (
        gp.notna()
        & wins.notna()
        & losses.notna()
        & ((wins + losses - gp).abs() > 0.01)
    )

    error_checks.append({
        "source": source_name,
        "colonne": "GP/W/L",
        "regle": "W + L = GP",
        "nombre_erreurs": int(mask.sum()),
        "exemples": "",
    })

In [14]:
# Pourcentages dont le domaine strict est compris entre 0 et 1

fraction_columns = {
    "player_bio": [
        "OREB_PCT",
        "DREB_PCT",
        "USG_PCT",
        "AST_PCT",
    ],

    "player_stats_base": [
        "W_PCT",
        "FG_PCT",
        "FG3_PCT",
        "FT_PCT",
    ],

    "player_stats_advanced": [
        "W_PCT",
        "OREB_PCT",
        "DREB_PCT",
        "REB_PCT",
        "AST_PCT",
        "USG_PCT",
        "E_USG_PCT",
    ],

    "team_stats_base": [
        "W_PCT",
        "FG_PCT",
        "FG3_PCT",
        "FT_PCT",
    ],

    "team_stats_advanced": [
        "W_PCT",
        "AST_PCT",
        "DREB_PCT",
        "REB_PCT",
        "TM_TOV_PCT",
    ],

    "kaggle_all_seasons": [
        "oreb_pct",
        "dreb_pct",
        "usg_pct",
        "ast_pct",
    ],
}


for source_name, columns in fraction_columns.items():
    df = sources_clean[source_name]

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        mask = (
            values.notna()
            & (
                (values < 0)
                | (values > 1)
            )
        )

        add_error_check(
            source_name,
            col,
            "Valeur comprise entre 0 et 1",
            mask,
            df
        )


# Pourcentages joueurs stockés sur une échelle 0 à 100

df = sources_clean["player_stats_advanced"]

for col in [
    "TM_TOV_PCT",
    "E_TOV_PCT",
]:
    if col not in df.columns:
        continue

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    mask = (
        values.notna()
        & (
            (values < 0)
            | (values > 100)
        )
    )

    add_error_check(
        "player_stats_advanced",
        col,
        "Valeur comprise entre 0 et 100",
        mask,
        df
    )

In [15]:
# Positions autorisées dans les rosters

rosters = sources_clean["rosters"]

allowed_positions = {
    "C",
    "C-F",
    "F",
    "F-C",
    "F-G",
    "G",
    "G-F",
}

if "POSITION" in rosters.columns:
    position_mask = (
        rosters["POSITION"].notna()
        & ~rosters["POSITION"].isin(
            allowed_positions
        )
    )

    add_error_check(
        "rosters",
        "POSITION",
        "Position appartenant à la liste autorisée",
        position_mask,
        rosters
    )


# Dates de naissance

if "BIRTH_DATE" in rosters.columns:
    birth_dates = pd.to_datetime(
        rosters["BIRTH_DATE"],
        format="%b %d, %Y",
        errors="coerce"
    )

    future_mask = (
        birth_dates.notna()
        & (
            birth_dates
            > pd.Timestamp.today().normalize()
        )
    )

    old_mask = (
        birth_dates.notna()
        & (
            birth_dates
            < pd.Timestamp("1940-01-01")
        )
    )

    add_error_check(
        "rosters",
        "BIRTH_DATE",
        "Aucune date de naissance future",
        future_mask,
        rosters
    )

    add_error_check(
        "rosters",
        "BIRTH_DATE",
        "Date de naissance supérieure ou égale à 1940-01-01",
        old_mask,
        rosters
    )

In [16]:
# Rapport des contrôles

obvious_errors_report = pd.DataFrame(
    error_checks
)

obvious_errors_report.to_csv(
    REPORT_DIR / "03_erreurs_evidentes_controles.csv",
    index=False,
    encoding="utf-8"
)

obvious_errors_found = obvious_errors_report[
    obvious_errors_report[
        "nombre_erreurs"
    ] > 0
]

print(
    "Contrôles réalisés :",
    len(obvious_errors_report)
)

print(
    "Contrôles avec erreurs :",
    len(obvious_errors_found)
)

print(
    "Violations détectées :",
    int(
        obvious_errors_report[
            "nombre_erreurs"
        ].sum()
    )
)

display(
    obvious_errors_found
)

Contrôles réalisés : 115
Contrôles avec erreurs : 0
Violations détectées : 0


,source,colonne,regle,nombre_erreurs,exemples


In [17]:
# Contrôle de la colonne technique Kaggle

kaggle = sources_clean[
    "kaggle_all_seasons"
]

if "Unnamed: 0" in kaggle.columns:

    values = pd.to_numeric(
        kaggle["Unnamed: 0"],
        errors="coerce"
    )

    expected = np.arange(
        len(kaggle)
    )

    is_export_index = (
        values.notna().all()
        and values.nunique() == len(kaggle)
        and np.array_equal(
            values.to_numpy(),
            expected
        )
    )

    print(
        "Index technique confirmé :",
        is_export_index
    )

else:
    is_export_index = False

    print(
        "Unnamed: 0 est déjà absente."
    )

Index technique confirmé : True


In [18]:
# Suppression de l'index technique

kaggle = sources_clean[
    "kaggle_all_seasons"
]

if (
    "Unnamed: 0" in kaggle.columns
    and is_export_index
):

    sources_clean[
        "kaggle_all_seasons"
    ] = kaggle.drop(
        columns=["Unnamed: 0"]
    ).copy()

    already_logged = any(
        row["source"] == "kaggle_all_seasons"
        and row["colonne"] == "Unnamed: 0"
        for row in cleaning_log_rows
    )

    if not already_logged:
        add_cleaning_log(
            source="kaggle_all_seasons",
            colonne="Unnamed: 0",
            probleme="Index technique issu de l'export CSV",
            action="Suppression de la colonne",
            lignes_affectees=len(kaggle),
            justification=(
                "La colonne contient uniquement l'index séquentiel "
                "des lignes et n'apporte aucune information métier."
            )
        )


print(
    sources_clean[
        "kaggle_all_seasons"
    ].shape
)

print(
    "Unnamed: 0" in
    sources_clean[
        "kaggle_all_seasons"
    ].columns
)

(12844, 21)
False


In [19]:
# Bilan avant et après l'étape 3

step3_summary_rows = []

for source_name, df in sources_clean.items():

    step3_summary_rows.append({
        "source": source_name,
        "lignes_avant":
            step3_rows_before[source_name],
        "lignes_apres":
            len(df),
        "variation_lignes":
            len(df)
            - step3_rows_before[source_name],
        "colonnes_avant":
            step3_cols_before[source_name],
        "colonnes_apres":
            df.shape[1],
        "variation_colonnes":
            df.shape[1]
            - step3_cols_before[source_name],
    })


step3_summary = pd.DataFrame(
    step3_summary_rows
)

step3_summary.to_csv(
    REPORT_DIR / "03_bilan_erreurs_evidentes.csv",
    index=False,
    encoding="utf-8"
)

display(step3_summary)

,source,lignes_avant,lignes_apres,variation_lignes,colonnes_avant,colonnes_apres,variation_colonnes
0,player_bio,12846,12846,0,25,25,0
1,player_stats_base,12846,12846,0,69,69,0
2,player_stats_advanced,12846,12846,0,81,81,0
3,team_stats_base,802,802,0,56,56,0
4,team_stats_advanced,802,802,0,48,48,0
5,rosters,11672,11672,0,18,18,0
6,kaggle_all_seasons,12844,12844,0,22,21,-1


In [20]:
cleaning_log = get_cleaning_log()

cleaning_log.to_csv(
    REPORT_DIR / "cleaning_log.csv",
    index=False,
    encoding="utf-8"
)

display(cleaning_log)

,#,source,colonne,probleme,action,lignes_affectees,justification
0,1,player_bio,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
1,2,player_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
2,3,player_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
3,4,team_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
4,5,team_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
5,6,rosters,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
6,7,kaggle_all_seasons,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
7,8,kaggle_all_seasons,Unnamed: 0,Index technique issu de l'export CSV,Suppression de la colonne,12844,La colonne contient uniquement l'index séquentiel des lignes et n'apporte aucune information métier.


### Résultat de la correction des erreurs évidentes

Les contrôles métier définis lors du diagnostic ont été réappliqués aux
données de travail.

Aucune valeur manifestement impossible n'a été détectée. Aucune ligne et
aucune valeur métier n'ont donc été supprimées ou corrigées.

La colonne `Unnamed: 0` du dataset Kaggle a été vérifiée comme étant un
simple index séquentiel issu de l'export CSV. Elle a été supprimée car elle
ne contient aucune information métier.

Le dataset Kaggle passe ainsi de 22 à 21 colonnes tout en conservant ses
12 844 lignes.

Les valeurs simplement extrêmes restent conservées. Leur traitement sera
décidé lors de l'étape consacrée aux outliers.

### Checklist : Erreurs évidentes

- [x] Supprimer les lignes avec des erreurs irréparables
  - Aucune ligne irréparable détectée
- [x] Corriger les valeurs corrigeables
  - Suppression de la colonne technique `Unnamed: 0`
- [x] Documenter chaque correction

### Bilan

- Erreurs métier manifestes détectées : **0**
- Lignes supprimées : **0**
- Valeurs métier corrigées : **0**
- Colonnes techniques supprimées : **1**
- Lignes Kaggle conservées : **12 844**

## 4. Correction des types de données

Cette étape vérifie et corrige les types de données lorsque la conversion
peut être réalisée sans perte d'information métier.

Les conversions forcées ne sont pas appliquées aux colonnes contenant
des valeurs métier textuelles, par exemple `Undrafted` ou `R`.

Aucune imputation statistique n'est réalisée.

In [21]:
# Types avant conversion

types_before_rows = []

for source_name, df in sources_clean.items():

    for col in df.columns:

        types_before_rows.append({
            "source": source_name,
            "colonne": col,
            "type_avant": str(df[col].dtype)
        })


types_before = pd.DataFrame(
    types_before_rows
)

types_before.to_csv(
    REPORT_DIR / "04_types_avant.csv",
    index=False,
    encoding="utf-8"
)

display(types_before)

,source,colonne,type_avant
0,player_bio,PLAYER_ID,int64
1,player_bio,PLAYER_NAME,string
2,player_bio,TEAM_ID,int64
3,player_bio,TEAM_ABBREVIATION,string
4,player_bio,AGE,float64
...,...,...,...
313,kaggle_all_seasons,dreb_pct,float64
314,kaggle_all_seasons,usg_pct,float64
315,kaggle_all_seasons,ts_pct,float64
316,kaggle_all_seasons,ast_pct,float64


In [22]:
# Recherche des colonnes texte contenant principalement des nombres

numeric_text_candidates = []

for source_name, df in sources_clean.items():

    for col in df.columns:

        if not (
            pd.api.types.is_object_dtype(df[col])
            or pd.api.types.is_string_dtype(df[col])
        ):
            continue

        values = (
            df[col]
            .astype("string")
            .str.strip()
        )

        non_missing = values[
            values.notna()
            & ~values.str.lower().isin(
                TEXT_MISSING_VALUES
            )
        ]

        if len(non_missing) == 0:
            continue

        converted = pd.to_numeric(
            non_missing,
            errors="coerce"
        )

        numeric_rate = (
            converted.notna().mean()
        )

        if numeric_rate >= 0.50:

            non_numeric_values = (
                non_missing[
                    converted.isna()
                ]
                .value_counts()
                .head(10)
                .index
                .tolist()
            )

            numeric_text_candidates.append({
                "source": source_name,
                "colonne": col,
                "type_actuel": str(df[col].dtype),
                "taux_convertible_pct": round(
                    numeric_rate * 100,
                    4
                ),
                "valeurs_non_numeriques": " | ".join(
                    map(str, non_numeric_values)
                )
            })


numeric_text_candidates = pd.DataFrame(
    numeric_text_candidates
)

display(numeric_text_candidates)

numeric_text_candidates.to_csv(
    REPORT_DIR / "04_candidats_numeriques_texte.csv",
    index=False,
    encoding="utf-8"
)

,source,colonne,type_actuel,taux_convertible_pct,valeurs_non_numeriques
0,player_bio,PLAYER_WEIGHT,string,100.0000,
1,player_bio,DRAFT_YEAR,string,82.1423,Undrafted
2,player_bio,DRAFT_ROUND,string,82.0093,Undrafted
3,player_bio,DRAFT_NUMBER,string,81.9965,Undrafted
4,rosters,EXP,string,83.7731,R
5,kaggle_all_seasons,draft_year,object,81.6412,Undrafted
6,kaggle_all_seasons,draft_round,object,81.2286,Undrafted
7,kaggle_all_seasons,draft_number,object,81.2052,Undrafted


In [23]:
# Conversion numérique sécurisée

def safe_numeric_conversion(
    source_name,
    col,
    target_type
):

    df = sources_clean[source_name]

    original = df[col].copy()

    text = (
        original
        .astype("string")
        .str.strip()
    )

    known_missing = (
        original.isna()
        | text.str.lower().isin(
            TEXT_MISSING_VALUES
        )
    )

    converted = pd.to_numeric(
        text,
        errors="coerce"
    )

    invalid_conversion = (
        ~known_missing
        & converted.isna()
    )

    invalid_count = int(
        invalid_conversion.sum()
    )

    if invalid_count > 0:

        print(
            f"{source_name}.{col} : "
            f"{invalid_count} valeur(s) non convertible(s)"
        )

        display(
            original.loc[
                invalid_conversion
            ]
            .value_counts()
            .head(20)
        )

        return False


    if target_type == "Int64":

        non_missing = converted.dropna()

        has_decimals = (
            (non_missing % 1 != 0)
            .any()
        )

        if has_decimals:

            print(
                f"{source_name}.{col} : "
                "des valeurs décimales empêchent "
                "la conversion en Int64"
            )

            return False

        converted = converted.astype(
            "Int64"
        )

    elif target_type == "Float64":

        converted = converted.astype(
            "Float64"
        )

    else:

        converted = converted.astype(
            target_type
        )


    sources_clean[
        source_name
    ][col] = converted

    print(
        f"{source_name}.{col} : "
        f"{original.dtype} -> "
        f"{converted.dtype}"
    )

    return True

In [24]:
# Conversions numériques certaines

numeric_conversions = {
    "player_bio": {
        "PLAYER_WEIGHT": "Float64",
    },

    "rosters": {
        "WEIGHT": "Float64",
        "AGE": "Int64",
        "NUM": "Int64",
    },
}


conversion_results = []


for source_name, columns in numeric_conversions.items():

    for col, target_type in columns.items():

        if col not in sources_clean[
            source_name
        ].columns:
            continue

        old_type = str(
            sources_clean[
                source_name
            ][col].dtype
        )

        success = safe_numeric_conversion(
            source_name,
            col,
            target_type
        )

        new_type = str(
            sources_clean[
                source_name
            ][col].dtype
        )

        conversion_results.append({
            "source": source_name,
            "colonne": col,
            "type_avant": old_type,
            "type_cible": target_type,
            "type_apres": new_type,
            "conversion_reussie": success,
        })

        if success and old_type != new_type:

            already_logged = any(
                row["source"] == source_name
                and row["colonne"] == col
                and row["probleme"] == "Type de données"
                for row in cleaning_log_rows
            )

            if not already_logged:

                add_cleaning_log(
                    source=source_name,
                    colonne=col,
                    probleme="Type de données",
                    action=f"Conversion vers {target_type}",
                    lignes_affectees=int(
                        sources_clean[
                            source_name
                        ][col]
                        .notna()
                        .sum()
                    ),
                    justification=(
                        "La colonne contient une information "
                        "numérique et la conversion a été validée "
                        "sans perte de valeur métier."
                    )
                )


conversion_report = pd.DataFrame(
    conversion_results
)

display(conversion_report)

player_bio.PLAYER_WEIGHT : string -> Float64
rosters.WEIGHT : int64 -> Float64
rosters.AGE : float64 -> Int64
rosters.NUM : float64 -> Int64


,source,colonne,type_avant,type_cible,type_apres,conversion_reussie
0,player_bio,PLAYER_WEIGHT,string,Float64,Float64,True
1,rosters,WEIGHT,int64,Float64,Float64,True
2,rosters,AGE,float64,Int64,Int64,True
3,rosters,NUM,float64,Int64,Int64,True


In [25]:
# Conversion de BIRTH_DATE en datetime

rosters = sources_clean["rosters"]

birth_before = rosters[
    "BIRTH_DATE"
].copy()

birth_after = pd.to_datetime(
    birth_before,
    format="%b %d, %Y",
    errors="coerce"
)

invalid_dates = (
    birth_before.notna()
    & birth_after.isna()
)

invalid_date_count = int(
    invalid_dates.sum()
)

print(
    "Dates non convertibles :",
    invalid_date_count
)


if invalid_date_count == 0:

    sources_clean[
        "rosters"
    ]["BIRTH_DATE"] = birth_after

    already_logged = any(
        row["source"] == "rosters"
        and row["colonne"] == "BIRTH_DATE"
        and row["probleme"] == "Type de données"
        for row in cleaning_log_rows
    )

    if not already_logged:

        add_cleaning_log(
            source="rosters",
            colonne="BIRTH_DATE",
            probleme="Type de données",
            action="Conversion vers datetime64[ns]",
            lignes_affectees=int(
                birth_before.notna().sum()
            ),
            justification=(
                "Toutes les dates respectent le format "
                "'%b %d, %Y' et sont convertibles sans erreur."
            )
        )


print(
    "Type final :",
    sources_clean[
        "rosters"
    ]["BIRTH_DATE"].dtype
)

Dates non convertibles : 0
Type final : datetime64[ns]


In [26]:
# Conversions différées

deferred_columns = {
    "player_bio": {
        "DRAFT_YEAR":
            "Contient la valeur métier Undrafted",
        "DRAFT_ROUND":
            "Contient la valeur métier Undrafted",
        "DRAFT_NUMBER":
            "Contient la valeur métier Undrafted",
    },

    "kaggle_all_seasons": {
        "draft_year":
            "Contient la valeur métier Undrafted",
        "draft_round":
            "Contient la valeur métier Undrafted",
        "draft_number":
            "Contient la valeur métier Undrafted",
    },

    "rosters": {
        "EXP":
            "Contient R pour Rookie",
        "HEIGHT":
            "Format pieds-pouces à uniformiser pendant la standardisation",
    },
}


deferred_rows = []


for source_name, columns in deferred_columns.items():

    for col, reason in columns.items():

        if col not in sources_clean[
            source_name
        ].columns:
            continue

        deferred_rows.append({
            "source": source_name,
            "colonne": col,
            "type_actuel": str(
                sources_clean[
                    source_name
                ][col].dtype
            ),
            "raison": reason,
        })


deferred_types = pd.DataFrame(
    deferred_rows
)

display(deferred_types)

deferred_types.to_csv(
    REPORT_DIR / "04_conversions_differees.csv",
    index=False,
    encoding="utf-8"
)

,source,colonne,type_actuel,raison
0,player_bio,DRAFT_YEAR,string,Contient la valeur métier Undrafted
1,player_bio,DRAFT_ROUND,string,Contient la valeur métier Undrafted
2,player_bio,DRAFT_NUMBER,string,Contient la valeur métier Undrafted
3,kaggle_all_seasons,draft_year,object,Contient la valeur métier Undrafted
4,kaggle_all_seasons,draft_round,object,Contient la valeur métier Undrafted
5,kaggle_all_seasons,draft_number,object,Contient la valeur métier Undrafted
6,rosters,EXP,string,Contient R pour Rookie
7,rosters,HEIGHT,string,Format pieds-pouces à uniformiser pendant la standardisation


In [27]:
# Types après conversion

types_after_rows = []

for source_name, df in sources_clean.items():

    for col in df.columns:

        types_after_rows.append({
            "source": source_name,
            "colonne": col,
            "type_apres": str(df[col].dtype)
        })


types_after = pd.DataFrame(
    types_after_rows
)


type_comparison = (
    types_before
    .merge(
        types_after,
        on=["source", "colonne"],
        how="outer"
    )
)


type_comparison[
    "type_modifie"
] = (
    type_comparison["type_avant"]
    != type_comparison["type_apres"]
)


types_changed = type_comparison[
    type_comparison[
        "type_modifie"
    ]
].copy()


display(types_changed)


type_comparison.to_csv(
    REPORT_DIR / "04_types_avant_apres.csv",
    index=False,
    encoding="utf-8"
)

,source,colonne,type_avant,type_apres,type_modifie
37,player_bio,PLAYER_WEIGHT,string,Float64,True
196,rosters,AGE,float64,Int64,True
197,rosters,BIRTH_DATE,string,datetime64[ns],True
203,rosters,NUM,float64,Int64,True
213,rosters,WEIGHT,int64,Float64,True


In [28]:
# Vérification des valeurs manquantes créées par les conversions

missing_conversion_rows = []

for source_name, df in sources_clean.items():

    raw_df = sources_raw[
        source_name
    ]

    common_columns = [
        col
        for col in df.columns
        if col in raw_df.columns
    ]

    for col in common_columns:

        before_missing = int(
            raw_df[col]
            .isna()
            .sum()
        )

        after_missing = int(
            df[col]
            .isna()
            .sum()
        )

        if before_missing != after_missing:

            missing_conversion_rows.append({
                "source": source_name,
                "colonne": col,
                "manquants_natifs_avant":
                    before_missing,
                "manquants_apres":
                    after_missing,
                "variation":
                    after_missing
                    - before_missing,
            })


missing_changes = pd.DataFrame(
    missing_conversion_rows
)

display(missing_changes)

,source,colonne,manquants_natifs_avant,manquants_apres,variation
0,player_bio,PLAYER_WEIGHT,6,71,65


In [29]:
# Vérification spécifique de PLAYER_WEIGHT

original_weight = sources_raw[
    "player_bio"
]["PLAYER_WEIGHT"]

weight_text = (
    original_weight
    .astype("string")
    .str.strip()
    .str.lower()
)

native_missing = int(
    original_weight
    .isna()
    .sum()
)

text_missing = int(
    (
        weight_text.isin(
            TEXT_MISSING_VALUES
        )
        & original_weight.notna()
    ).sum()
)

final_missing = int(
    sources_clean[
        "player_bio"
    ]["PLAYER_WEIGHT"]
    .isna()
    .sum()
)


print(
    "NaN natifs :",
    native_missing
)

print(
    "Marqueurs textuels d'absence :",
    text_missing
)

print(
    "NaN après conversion :",
    final_missing
)

print(
    "Cohérence :",
    native_missing
    + text_missing
    == final_missing
)

NaN natifs : 6
Marqueurs textuels d'absence : 65
NaN après conversion : 71
Cohérence : True


In [30]:
# Validation de l'étape 4

failed_conversions = conversion_report[
    conversion_report[
        "conversion_reussie"
    ] == False
]


validation_step4 = pd.DataFrame({
    "controle": [
        "Conversions numériques échouées",
        "Dates non convertibles",
        "Nombre de lignes modifié",
    ],

    "valeur": [
        len(failed_conversions),
        invalid_date_count,
        sum(
            len(sources_clean[source])
            != step3_rows_before[source]
            for source in sources_clean
        ),
    ]
})


display(validation_step4)


assert len(failed_conversions) == 0
assert invalid_date_count == 0


print(
    "Toutes les conversions de types sont valides."
)

,controle,valeur
0,Conversions numériques échouées,0
1,Dates non convertibles,0
2,Nombre de lignes modifié,0


Toutes les conversions de types sont valides.


In [ ]:
# Mise à jour du log

cleaning_log = get_cleaning_log()

cleaning_log.to_csv(
    REPORT_DIR / "cleaning_log.csv",
    index=False,
    encoding="utf-8"
)

display(cleaning_log)

### Résultat de la correction des types

Les types de données ont été vérifiés sur les sept sources.

Les colonnes numériques stockées sous une représentation textuelle ont été
converties uniquement lorsque la conversion était possible sans perte
d'information métier.

`player_bio.PLAYER_WEIGHT` a été convertie en type numérique. Les marqueurs
textuels représentant une absence de valeur ont été transformés en valeurs
manquantes natives.

`rosters.BIRTH_DATE` a été convertie en `datetime64[ns]`. Toutes les dates
ont été converties sans erreur.

Les colonnes de draft contenant `Undrafted`, ainsi que `rosters.EXP`
contenant `R` et `rosters.HEIGHT` utilisant un format pieds-pouces, restent
volontairement sous forme textuelle. Leur traitement nécessite une
standardisation métier et sera réalisé à l'étape suivante.

Aucune ligne et aucune valeur métier valide n'ont été perdues pendant les
conversions.

### Checklist : Types de données

- [x] Convertir les dates en datetime
- [x] Convertir les numériques en int/float lorsque la conversion est certaine
- [x] Vérifier les conversions et l'absence d'erreurs

## 5. Standardisation

Cette étape uniformise les formats et conventions identifiés pendant
le diagnostic de qualité.

Les transformations portent sur :

- les espaces superflus ;
- la casse des codes catégoriels ;
- les variantes orthographiques certaines ;
- la représentation de l'expérience NBA ;
- les unités de taille et de poids.

Les mappings ambigus ne sont pas appliqués automatiquement.

Les colonnes originales sont conservées lorsqu'une conversion d'unité crée
une nouvelle représentation afin de préserver la traçabilité.

In [31]:
# Etat avant standardisation

step5_rows_before = {
    source: len(df)
    for source, df in sources_clean.items()
}

step5_cols_before = {
    source: df.shape[1]
    for source, df in sources_clean.items()
}

standardization_rows = []


def add_standardization_report(
    source,
    colonne,
    transformation,
    lignes_affectees
):
    standardization_rows.append({
        "source": source,
        "colonne": colonne,
        "transformation": transformation,
        "lignes_affectees": int(lignes_affectees),
    })


def log_step5_once(
    source,
    colonne,
    probleme,
    action,
    lignes_affectees,
    justification
):
    already_logged = any(
        row["source"] == source
        and row["colonne"] == colonne
        and row["action"] == action
        for row in cleaning_log_rows
    )

    if not already_logged:
        add_cleaning_log(
            source=source,
            colonne=colonne,
            probleme=probleme,
            action=action,
            lignes_affectees=lignes_affectees,
            justification=justification
        )

In [32]:
# Suppression des espaces externes

for source_name, df in sources_clean.items():

    for col in df.columns:

        series = df[col]

        if isinstance(series.dtype, pd.StringDtype):

            before = series.copy()
            after = series.str.strip()

        elif pd.api.types.is_object_dtype(series):

            before = series.copy()

            after = series.map(
                lambda value:
                    value.strip()
                    if isinstance(value, str)
                    else value
            )

        else:
            continue

        before_cmp = (
            before
            .astype("string")
            .fillna("<NA>")
        )

        after_cmp = (
            after
            .astype("string")
            .fillna("<NA>")
        )

        changed = int(
            (before_cmp != after_cmp).sum()
        )

        if changed > 0:

            sources_clean[
                source_name
            ][col] = after

            add_standardization_report(
                source_name,
                col,
                "Suppression des espaces externes",
                changed
            )

            log_step5_once(
                source=source_name,
                colonne=col,
                probleme="Espaces superflus",
                action="Application de strip()",
                lignes_affectees=changed,
                justification=(
                    "Les espaces en début ou fin de chaîne "
                    "n'apportent aucune information métier."
                )
            )


space_changes = pd.DataFrame(
    [
        row
        for row in standardization_rows
        if row["transformation"]
        == "Suppression des espaces externes"
    ]
)

display(space_changes)

,source,colonne,transformation,lignes_affectees
0,rosters,SCHOOL,Suppression des espaces externes,2
1,kaggle_all_seasons,college,Suppression des espaces externes,7


In [33]:
# Standardisation de la casse des codes

case_columns = {
    "player_bio": [
        "TEAM_ABBREVIATION",
    ],

    "player_stats_base": [
        "TEAM_ABBREVIATION",
    ],

    "player_stats_advanced": [
        "TEAM_ABBREVIATION",
    ],

    "team_stats_base": [
        "TEAM_ABBREVIATION",
    ],

    "team_stats_advanced": [
        "TEAM_ABBREVIATION",
    ],

    "rosters": [
        "POSITION",
    ],

    "kaggle_all_seasons": [
        "team_abbreviation",
    ],
}


for source_name, columns in case_columns.items():

    df = sources_clean[source_name]

    for col in columns:

        if col not in df.columns:
            continue

        before = df[col].copy()

        after = before.map(
            lambda value:
                value.upper()
                if isinstance(value, str)
                else value
        )

        before_cmp = (
            before.astype("string")
            .fillna("<NA>")
        )

        after_cmp = (
            after.astype("string")
            .fillna("<NA>")
        )

        changed = int(
            (before_cmp != after_cmp).sum()
        )

        sources_clean[
            source_name
        ][col] = after

        if changed > 0:

            add_standardization_report(
                source_name,
                col,
                "Standardisation en majuscules",
                changed
            )

            log_step5_once(
                source=source_name,
                colonne=col,
                probleme="Casse non uniforme",
                action="Conversion en majuscules",
                lignes_affectees=changed,
                justification=(
                    "La colonne contient un code catégoriel "
                    "dont la casse doit être uniforme."
                )
            )

In [34]:
# Mappings orthographiques certains

nickname_mapping = {
    "Dajuan": "DaJuan",
    "Deandre": "DeAndre",
    "DeAndre'": "DeAndre",
    "Danté": "Dante",
}


for source_name, df in sources_clean.items():

    if "NICKNAME" not in df.columns:
        continue

    before = df["NICKNAME"].copy()

    after = before.replace(
        nickname_mapping
    )

    changed = int(
        (
            before.astype("string").fillna("<NA>")
            !=
            after.astype("string").fillna("<NA>")
        ).sum()
    )

    sources_clean[
        source_name
    ]["NICKNAME"] = after

    if changed > 0:

        add_standardization_report(
            source_name,
            "NICKNAME",
            "Mapping de variantes orthographiques",
            changed
        )

        log_step5_once(
            source=source_name,
            colonne="NICKNAME",
            probleme="Variantes orthographiques",
            action="Mapping vers une écriture unique",
            lignes_affectees=changed,
            justification=(
                "Les variantes avaient été identifiées "
                "comme représentant le même libellé."
            )
        )


college_mapping = {
    "Indiana Purdue-Indianapolis":
        "Indiana-Purdue Indianapolis"
}


for source_name, col in [
    ("player_bio", "COLLEGE"),
    ("rosters", "SCHOOL"),
    ("kaggle_all_seasons", "college"),
]:

    if col not in sources_clean[
        source_name
    ].columns:
        continue

    before = sources_clean[
        source_name
    ][col].copy()

    after = before.replace(
        college_mapping
    )

    changed = int(
        (
            before.astype("string").fillna("<NA>")
            !=
            after.astype("string").fillna("<NA>")
        ).sum()
    )

    sources_clean[
        source_name
    ][col] = after

    if changed > 0:

        add_standardization_report(
            source_name,
            col,
            "Mapping d'une variante d'établissement",
            changed
        )

        log_step5_once(
            source=source_name,
            colonne=col,
            probleme="Variation orthographique",
            action=(
                "Mapping vers "
                "Indiana-Purdue Indianapolis"
            ),
            lignes_affectees=changed,
            justification=(
                "Les deux écritures avaient été identifiées "
                "comme une même valeur lors du diagnostic."
            )
        )

In [35]:
# Standardisation du libellé Undrafted

draft_columns = {
    "player_bio": [
        "DRAFT_YEAR",
        "DRAFT_ROUND",
        "DRAFT_NUMBER",
    ],

    "kaggle_all_seasons": [
        "draft_year",
        "draft_round",
        "draft_number",
    ],
}


for source_name, columns in draft_columns.items():

    for col in columns:

        if col not in sources_clean[
            source_name
        ].columns:
            continue

        before = sources_clean[
            source_name
        ][col].copy()

        after = before.map(
            lambda value:
                "Undrafted"
                if (
                    isinstance(value, str)
                    and value.strip().lower()
                    == "undrafted"
                )
                else value
        )

        changed = int(
            (
                before.astype("string").fillna("<NA>")
                !=
                after.astype("string").fillna("<NA>")
            ).sum()
        )

        sources_clean[
            source_name
        ][col] = after

        if changed > 0:

            add_standardization_report(
                source_name,
                col,
                "Standardisation du libellé Undrafted",
                changed
            )

In [36]:
# Standardisation de l'expérience NBA

rosters = sources_clean["rosters"]

exp_text = (
    rosters["EXP"]
    .astype("string")
    .str.strip()
)


exp_standard = exp_text.replace({
    "R": "0",
    "r": "0",
})


exp_years = pd.to_numeric(
    exp_standard,
    errors="coerce"
)


invalid_exp = (
    exp_text.notna()
    & exp_years.isna()
)


print(
    "Valeurs EXP non convertibles :",
    int(invalid_exp.sum())
)


if invalid_exp.any():

    display(
        exp_text[
            invalid_exp
        ]
        .value_counts()
        .head(20)
    )

else:

    sources_clean[
        "rosters"
    ]["EXP_YEARS"] = (
        exp_years.astype("Int64")
    )

    rookie_count = int(
        exp_text.str.upper()
        .eq("R")
        .sum()
    )

    add_standardization_report(
        "rosters",
        "EXP_YEARS",
        "Rookie R converti en 0 année d'expérience",
        rookie_count
    )

    log_step5_once(
        source="rosters",
        colonne="EXP_YEARS",
        probleme="Format mixte de l'expérience",
        action=(
            "Création d'une représentation numérique "
            "avec R = 0"
        ),
        lignes_affectees=int(
            exp_years.notna().sum()
        ),
        justification=(
            "La valeur R représente un rookie. "
            "La représentation standardisée correspond "
            "à 0 année d'expérience NBA antérieure."
        )
    )


display(
    sources_clean[
        "rosters"
    ][
        ["EXP", "EXP_YEARS"]
    ]
    .drop_duplicates()
    .sort_values(
        "EXP_YEARS"
    )
)

Valeurs EXP non convertibles : 0


,EXP,EXP_YEARS
3,R,0
5,1,1
0,2,2
13,3,3
4,4,4
1,5,5
16,6,6
2,7,7
17,8,8
12,9,9


In [37]:
# Standardisation des unités taille et poids

LB_TO_KG = 0.453592
INCH_TO_CM = 2.54


# player_bio

player_bio = sources_clean[
    "player_bio"
]

player_bio[
    "PLAYER_HEIGHT_CM"
] = (
    pd.to_numeric(
        player_bio[
            "PLAYER_HEIGHT_INCHES"
        ],
        errors="coerce"
    )
    * INCH_TO_CM
).astype("Float64")


player_bio[
    "PLAYER_WEIGHT_KG"
] = (
    pd.to_numeric(
        player_bio[
            "PLAYER_WEIGHT"
        ],
        errors="coerce"
    )
    * LB_TO_KG
).astype("Float64")


# Kaggle

kaggle = sources_clean[
    "kaggle_all_seasons"
]

kaggle[
    "PLAYER_HEIGHT_CM"
] = (
    pd.to_numeric(
        kaggle[
            "player_height"
        ],
        errors="coerce"
    )
    .astype("Float64")
)

kaggle[
    "PLAYER_WEIGHT_KG"
] = (
    pd.to_numeric(
        kaggle[
            "player_weight"
        ],
        errors="coerce"
    )
    .astype("Float64")
)

In [38]:
# Conversion de HEIGHT du roster en centimètres

rosters = sources_clean[
    "rosters"
]

height_text = (
    rosters["HEIGHT"]
    .astype("string")
    .str.strip()
)


height_parts = height_text.str.extract(
    r"^(\d+)-(\d+)$"
)


height_missing = (
    height_text.isna()
    | height_text.str.lower().isin(
        TEXT_MISSING_VALUES
    )
)


height_invalid = (
    ~height_missing
    & (
        height_parts[0].isna()
        | height_parts[1].isna()
    )
)


print(
    "Tailles roster non convertibles :",
    int(height_invalid.sum())
)


if height_invalid.any():

    display(
        height_text[
            height_invalid
        ]
        .value_counts()
        .head(20)
    )

else:

    feet = pd.to_numeric(
        height_parts[0],
        errors="coerce"
    )

    inches = pd.to_numeric(
        height_parts[1],
        errors="coerce"
    )

    total_inches = (
        feet * 12
        + inches
    )

    rosters[
        "PLAYER_HEIGHT_CM"
    ] = (
        total_inches
        * INCH_TO_CM
    ).astype("Float64")


    rosters[
        "PLAYER_WEIGHT_KG"
    ] = (
        pd.to_numeric(
            rosters["WEIGHT"],
            errors="coerce"
        )
        * LB_TO_KG
    ).astype("Float64")

Tailles roster non convertibles : 0


In [39]:
# Journalisation des conversions d'unités

unit_logs = [
    (
        "player_bio",
        "PLAYER_HEIGHT_CM",
        int(
            sources_clean[
                "player_bio"
            ][
                "PLAYER_HEIGHT_CM"
            ].notna().sum()
        ),
        "Conversion des pouces vers les centimètres"
    ),

    (
        "player_bio",
        "PLAYER_WEIGHT_KG",
        int(
            sources_clean[
                "player_bio"
            ][
                "PLAYER_WEIGHT_KG"
            ].notna().sum()
        ),
        "Conversion des livres vers les kilogrammes"
    ),

    (
        "rosters",
        "PLAYER_HEIGHT_CM",
        int(
            sources_clean[
                "rosters"
            ][
                "PLAYER_HEIGHT_CM"
            ].notna().sum()
        ),
        "Conversion du format pieds-pouces vers les centimètres"
    ),

    (
        "rosters",
        "PLAYER_WEIGHT_KG",
        int(
            sources_clean[
                "rosters"
            ][
                "PLAYER_WEIGHT_KG"
            ].notna().sum()
        ),
        "Conversion des livres vers les kilogrammes"
    ),

    (
        "kaggle_all_seasons",
        "PLAYER_HEIGHT_CM",
        int(
            sources_clean[
                "kaggle_all_seasons"
            ][
                "PLAYER_HEIGHT_CM"
            ].notna().sum()
        ),
        "Reprise de la taille déjà exprimée en centimètres"
    ),

    (
        "kaggle_all_seasons",
        "PLAYER_WEIGHT_KG",
        int(
            sources_clean[
                "kaggle_all_seasons"
            ][
                "PLAYER_WEIGHT_KG"
            ].notna().sum()
        ),
        "Reprise du poids déjà exprimé en kilogrammes"
    ),
]


for (
    source_name,
    col,
    affected,
    justification
) in unit_logs:

    log_step5_once(
        source=source_name,
        colonne=col,
        probleme="Unités hétérogènes entre sources",
        action="Standardisation de l'unité",
        lignes_affectees=affected,
        justification=justification
    )

In [40]:
# Mappings nécessitant encore une décision métier

deferred_mappings = pd.DataFrame([
    {
        "source": "player_bio",
        "colonne": "COUNTRY",
        "valeur": "DRC",
        "candidat": "Democratic Republic of the Congo",
        "decision": "Non modifié",
        "raison": (
            "Correspondance détectée pendant le diagnostic "
            "mais marquée comme nécessitant une vérification."
        ),
    },

    {
        "source": "kaggle_all_seasons",
        "colonne": "country",
        "valeur": "DRC",
        "candidat": "Democratic Republic of the Congo",
        "decision": "Non modifié",
        "raison": (
            "Correspondance détectée pendant le diagnostic "
            "mais marquée comme nécessitant une vérification."
        ),
    },

    {
        "source": "plusieurs",
        "colonne": "PLAYER_NAME / player_name",
        "valeur": (
            "Variantes d'accents et d'orthographe"
        ),
        "candidat": "Aucun mapping forcé",
        "decision": "Non modifié",
        "raison": (
            "PLAYER_ID sera utilisé comme identifiant fiable. "
            "Les noms ne doivent pas être altérés inutilement."
        ),
    },

    {
        "source": "plusieurs",
        "colonne": "COLLEGE / SCHOOL / college",
        "valeur": "Différences inter-sources restantes",
        "candidat": "Mappings à confirmer au cas par cas",
        "decision": "Non modifié",
        "raison": (
            "Certaines différences correspondent à de vrais "
            "établissements distincts ou à des clubs étrangers."
        ),
    },
])


deferred_mappings.to_csv(
    REPORT_DIR
    / "05_mappings_differes.csv",
    index=False,
    encoding="utf-8"
)

display(deferred_mappings)

,source,colonne,valeur,candidat,decision,raison
0,player_bio,COUNTRY,DRC,Democratic Republic of the Congo,Non modifié,Correspondance détectée pendant le diagnostic mais marquée comme nécessitant une vérification.
1,kaggle_all_seasons,country,DRC,Democratic Republic of the Congo,Non modifié,Correspondance détectée pendant le diagnostic mais marquée comme nécessitant une vérification.
2,plusieurs,PLAYER_NAME / player_name,Variantes d'accents et d'orthographe,Aucun mapping forcé,Non modifié,PLAYER_ID sera utilisé comme identifiant fiable. Les noms ne doivent pas être altérés inutilement.
3,plusieurs,COLLEGE / SCHOOL / college,Différences inter-sources restantes,Mappings à confirmer au cas par cas,Non modifié,Certaines différences correspondent à de vrais établissements distincts ou à des clubs étrangers.


In [41]:
# Vérification des espaces externes restants

remaining_space_rows = []


for source_name, df in sources_clean.items():

    for col in df.columns:

        series = df[col]

        if not (
            isinstance(
                series.dtype,
                pd.StringDtype
            )
            or pd.api.types.is_object_dtype(
                series
            )
        ):
            continue

        values = series.map(
            lambda value:
                value
                if isinstance(value, str)
                else pd.NA
        )

        mask = (
            values.notna()
            & (
                values
                != values.str.strip()
            )
        )

        if mask.any():

            remaining_space_rows.append({
                "source": source_name,
                "colonne": col,
                "nombre": int(
                    mask.sum()
                ),
            })


remaining_spaces = pd.DataFrame(
    remaining_space_rows
)


print(
    "Colonnes avec espaces externes restants :",
    len(remaining_spaces)
)

display(remaining_spaces)

Colonnes avec espaces externes restants : 0


""


In [42]:
# Validation des conversions d'unités

unit_validation = []


pb = sources_clean["player_bio"]

expected_pb_height = (
    pd.to_numeric(
        pb["PLAYER_HEIGHT_INCHES"],
        errors="coerce"
    )
    * INCH_TO_CM
)

expected_pb_weight = (
    pd.to_numeric(
        pb["PLAYER_WEIGHT"],
        errors="coerce"
    )
    * LB_TO_KG
)


unit_validation.append({
    "controle":
        "player_bio hauteur",
    "valide":
        np.allclose(
            pb["PLAYER_HEIGHT_CM"]
            .astype(float),
            expected_pb_height,
            equal_nan=True
        )
})

unit_validation.append({
    "controle":
        "player_bio poids",
    "valide":
        np.allclose(
            pb["PLAYER_WEIGHT_KG"]
            .astype(float),
            expected_pb_weight,
            equal_nan=True
        )
})


kaggle = sources_clean[
    "kaggle_all_seasons"
]

unit_validation.append({
    "controle":
        "Kaggle hauteur",
    "valide":
        np.allclose(
            kaggle["PLAYER_HEIGHT_CM"]
            .astype(float),
            pd.to_numeric(
                kaggle["player_height"],
                errors="coerce"
            ),
            equal_nan=True
        )
})

unit_validation.append({
    "controle":
        "Kaggle poids",
    "valide":
        np.allclose(
            kaggle["PLAYER_WEIGHT_KG"]
            .astype(float),
            pd.to_numeric(
                kaggle["player_weight"],
                errors="coerce"
            ),
            equal_nan=True
        )
})


unit_validation = pd.DataFrame(
    unit_validation
)

display(unit_validation)

assert unit_validation[
    "valide"
].all()

,controle,valide
0,player_bio hauteur,True
1,player_bio poids,True
2,Kaggle hauteur,True
3,Kaggle poids,True


In [43]:
# Rapport de standardisation

standardization_report = pd.DataFrame(
    standardization_rows
)


standardization_report.to_csv(
    REPORT_DIR
    / "05_standardisation.csv",
    index=False,
    encoding="utf-8"
)


step5_summary_rows = []


for source_name, df in sources_clean.items():

    step5_summary_rows.append({
        "source": source_name,
        "lignes_avant":
            step5_rows_before[source_name],
        "lignes_apres":
            len(df),
        "variation_lignes":
            len(df)
            - step5_rows_before[source_name],
        "colonnes_avant":
            step5_cols_before[source_name],
        "colonnes_apres":
            df.shape[1],
        "variation_colonnes":
            df.shape[1]
            - step5_cols_before[source_name],
    })


step5_summary = pd.DataFrame(
    step5_summary_rows
)


step5_summary.to_csv(
    REPORT_DIR
    / "05_bilan_standardisation.csv",
    index=False,
    encoding="utf-8"
)


display(
    standardization_report
)

display(
    step5_summary
)

,source,colonne,transformation,lignes_affectees
0,rosters,SCHOOL,Suppression des espaces externes,2
1,kaggle_all_seasons,college,Suppression des espaces externes,7
2,player_stats_base,NICKNAME,Mapping de variantes orthographiques,15
3,player_stats_advanced,NICKNAME,Mapping de variantes orthographiques,15
4,rosters,NICKNAME,Mapping de variantes orthographiques,19
5,kaggle_all_seasons,college,Mapping d'une variante d'établissement,9
6,rosters,EXP_YEARS,Rookie R converti en 0 année d'expérience,1894


,source,lignes_avant,lignes_apres,variation_lignes,colonnes_avant,colonnes_apres,variation_colonnes
0,player_bio,12846,12846,0,25,27,2
1,player_stats_base,12846,12846,0,69,69,0
2,player_stats_advanced,12846,12846,0,81,81,0
3,team_stats_base,802,802,0,56,56,0
4,team_stats_advanced,802,802,0,48,48,0
5,rosters,11672,11672,0,18,21,3
6,kaggle_all_seasons,12844,12844,0,21,23,2


In [44]:
# Validation de l'étape 5

step5_validation = pd.DataFrame({
    "controle": [
        "Lignes supprimées",
        "Espaces externes restants",
        "EXP non convertible",
        "HEIGHT roster non convertible",
        "Conversions d'unités invalides",
    ],

    "valeur": [
        sum(
            step5_rows_before[source]
            - len(sources_clean[source])
            for source in sources_clean
        ),

        len(remaining_spaces),

        int(invalid_exp.sum()),

        int(height_invalid.sum()),

        int(
            (~unit_validation["valide"])
            .sum()
        ),
    ]
})


display(step5_validation)


assert (
    step5_validation[
        "valeur"
    ] == 0
).all()


print(
    "Tous les contrôles de standardisation sont valides."
)

,controle,valeur
0,Lignes supprimées,0
1,Espaces externes restants,0
2,EXP non convertible,0
3,HEIGHT roster non convertible,0
4,Conversions d'unités invalides,0


Tous les contrôles de standardisation sont valides.


In [45]:
# Mise à jour du log de nettoyage

cleaning_log = get_cleaning_log()

cleaning_log.to_csv(
    REPORT_DIR / "cleaning_log.csv",
    index=False,
    encoding="utf-8"
)

display(cleaning_log)

,#,source,colonne,probleme,action,lignes_affectees,justification
0,1,player_bio,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
1,2,player_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
2,3,player_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
3,4,team_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
4,5,team_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
5,6,rosters,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
6,7,kaggle_all_seasons,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
7,8,kaggle_all_seasons,Unnamed: 0,Index technique issu de l'export CSV,Suppression de la colonne,12844,La colonne contient uniquement l'index séquentiel des lignes et n'apporte aucune information métier.
8,9,player_bio,PLAYER_WEIGHT,Type de données,Conversion vers Float64,12775,La colonne contient une information numérique et la conversion a été validée sans perte de valeur métier.
9,10,rosters,WEIGHT,Type de données,Conversion vers Float64,11672,La colonne contient une information numérique et la conversion a été validée sans perte de valeur métier.


### Résultat de la standardisation

Les conventions de représentation ont été uniformisées sur les sept sources.

Les espaces externes présents dans certaines colonnes texte ont été supprimés.
La casse des codes catégoriels, notamment les abréviations d'équipes et les
positions, a été uniformisée.

Les variantes orthographiques certaines identifiées pendant le diagnostic ont
été regroupées à l'aide de mappings explicites. Les valeurs ambiguës n'ont pas
été modifiées automatiquement.

Le libellé `Undrafted` est conservé comme information métier et utilise une
écriture commune.

Une représentation numérique de l'expérience a été créée dans `EXP_YEARS`,
avec `R` représentant un rookie et correspondant à 0 année d'expérience
antérieure.

Les unités physiques ont été harmonisées en utilisant les centimètres pour la
taille et les kilogrammes pour le poids. Les colonnes standardisées
`PLAYER_HEIGHT_CM` et `PLAYER_WEIGHT_KG` sont disponibles dans les trois
sources contenant ces informations.

Les colonnes sources sont conservées afin de préserver la traçabilité.

Aucune ligne n'a été supprimée pendant cette étape.

Les mappings inter-sources dont l'équivalence n'est pas suffisamment certaine
restent documentés comme décisions différées.

### Checklist : Standardisation

- [x] Standardiser la casse des colonnes texte
- [x] Appliquer les mappings de valeurs certaines
- [x] Supprimer les espaces superflus
- [x] Uniformiser les formats et unités

## 6. Traitement des valeurs manquantes

Les valeurs manquantes sont traitées selon les stratégies définies pendant
le diagnostic de qualité.

Aucune imputation statistique n'est réalisée dans cette phase afin d'éviter
toute fuite de données.

Les décisions retenues sont les suivantes :

- suppression des colonnes très incomplètes et peu exploitables ;
- conservation des observations valides ;
- création d'indicateurs de valeurs manquantes lorsque cette absence peut
  porter une information utile ;
- conservation des `NaN` lorsque leur éventuelle imputation devra être
  réalisée ultérieurement dans la pipeline ML.

Aucune ligne n'est supprimée uniquement à cause d'une information secondaire
manquante.

In [46]:
# Etat avant traitement des valeurs manquantes

step6_rows_before = {
    source: len(df)
    for source, df in sources_clean.items()
}

step6_cols_before = {
    source: df.shape[1]
    for source, df in sources_clean.items()
}


# Normalisation des marqueurs textuels d'absence

MISSING_TEXT_MARKERS = {
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "<na>",
}

missing_marker_changes = []


for source_name, df in sources_clean.items():

    for col in df.columns:

        series = df[col]

        if not (
            isinstance(series.dtype, pd.StringDtype)
            or pd.api.types.is_object_dtype(series)
        ):
            continue

        text = (
            series
            .astype("string")
            .str.strip()
            .str.lower()
        )

        marker_mask = (
            series.notna()
            & text.isin(
                MISSING_TEXT_MARKERS
            )
        )

        count = int(
            marker_mask.sum()
        )

        if count == 0:
            continue

        sources_clean[
            source_name
        ].loc[
            marker_mask,
            col
        ] = pd.NA

        missing_marker_changes.append({
            "source": source_name,
            "colonne": col,
            "marqueurs_convertis": count,
        })

        log_step5_once(
            source=source_name,
            colonne=col,
            probleme="Valeurs manquantes représentées sous forme textuelle",
            action="Conversion des marqueurs textuels vers pd.NA",
            lignes_affectees=count,
            justification=(
                "Les valeurs ne représentent pas une information métier "
                "et sont normalisées comme valeurs manquantes natives."
            )
        )


missing_marker_changes = pd.DataFrame(
    missing_marker_changes
)

display(missing_marker_changes)

,source,colonne,marqueurs_convertis
0,rosters,SCHOOL,2
1,kaggle_all_seasons,college,5


In [47]:
# Valeurs manquantes après normalisation des marqueurs

missing_before_rows = []


for source_name, df in sources_clean.items():

    for col in df.columns:

        count = int(
            df[col].isna().sum()
        )

        if count == 0:
            continue

        missing_before_rows.append({
            "source": source_name,
            "colonne": col,
            "manquants": count,
            "pourcentage": round(
                count / len(df) * 100,
                4
            ),
        })


missing_before_step6 = (
    pd.DataFrame(
        missing_before_rows
    )
    .sort_values(
        ["pourcentage", "source"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


missing_before_step6.to_csv(
    REPORT_DIR / "06_manquants_avant.csv",
    index=False,
    encoding="utf-8"
)


display(
    missing_before_step6
)

,source,colonne,manquants,pourcentage
0,rosters,HOW_ACQUIRED,11280,96.6415
1,rosters,SUPPLEMENTAL_STATUS,5843,50.0600
2,kaggle_all_seasons,college,1859,14.4737
3,player_bio,COLLEGE,1821,14.1756
4,player_bio,DRAFT_NUMBER,104,0.8096
5,player_bio,DRAFT_ROUND,95,0.7395
6,player_bio,PLAYER_HEIGHT,71,0.5527
7,player_bio,PLAYER_HEIGHT_INCHES,71,0.5527
8,player_bio,PLAYER_WEIGHT,71,0.5527
9,player_bio,PLAYER_HEIGHT_CM,71,0.5527


In [48]:
# Stratégies décidées pour les colonnes manquantes

strategy_map = {
    ("rosters", "HOW_ACQUIRED"): {
        "strategie": "Suppression de colonne",
        "flag": None,
        "justification": (
            "Colonne manquante sur environ 97 % des lignes, "
            "absence fortement systématique et faible intérêt "
            "pour la prédiction des performances."
        ),
    },

    ("rosters", "SUPPLEMENTAL_STATUS"): {
        "strategie": "Suppression de colonne",
        "flag": None,
        "justification": (
            "Plus de 50 % de valeurs manquantes et absence liée "
            "à une évolution du schéma de la source roster."
        ),
    },

    ("player_bio", "COLLEGE"): {
        "strategie": "Flag + conservation",
        "flag": "COLLEGE_IS_MISSING",
        "justification": (
            "Environ 14 % de valeurs manquantes. "
            "L'absence n'est pas suffisamment aléatoire pour supprimer "
            "les joueurs concernés."
        ),
    },

    ("kaggle_all_seasons", "college"): {
        "strategie": "Flag + conservation",
        "flag": "COLLEGE_IS_MISSING",
        "justification": (
            "Environ 14 % de valeurs manquantes. "
            "Les lignes restent utiles pour l'analyse et le ML."
        ),
    },

    ("player_bio", "DRAFT_YEAR"): {
        "strategie": "Flag + conservation",
        "flag": "DRAFT_INFO_IS_MISSING",
        "justification": (
            "L'information de draft est conservée sans imputation."
        ),
    },

    ("player_bio", "DRAFT_ROUND"): {
        "strategie": "Flag + conservation",
        "flag": "DRAFT_INFO_IS_MISSING",
        "justification": (
            "L'information de draft est conservée sans imputation."
        ),
    },

    ("player_bio", "DRAFT_NUMBER"): {
        "strategie": "Flag + conservation",
        "flag": "DRAFT_INFO_IS_MISSING",
        "justification": (
            "L'information de draft est conservée sans imputation."
        ),
    },

    ("player_bio", "PLAYER_HEIGHT"): {
        "strategie": "Flag + conservation",
        "flag": "PLAYER_HEIGHT_IS_MISSING",
        "justification": (
            "La ligne joueur reste exploitable malgré une taille absente."
        ),
    },

    ("player_bio", "PLAYER_HEIGHT_INCHES"): {
        "strategie": "Flag + conservation",
        "flag": "PLAYER_HEIGHT_IS_MISSING",
        "justification": (
            "La ligne joueur reste exploitable malgré une taille absente."
        ),
    },

    ("player_bio", "PLAYER_HEIGHT_CM"): {
        "strategie": "Flag + conservation",
        "flag": "PLAYER_HEIGHT_IS_MISSING",
        "justification": (
            "La représentation standardisée conserve la valeur manquante."
        ),
    },

    ("player_bio", "PLAYER_WEIGHT"): {
        "strategie": "Flag + conservation",
        "flag": "PLAYER_WEIGHT_IS_MISSING",
        "justification": (
            "Le poids n'est pas imputé avant la séparation train/test."
        ),
    },

    ("player_bio", "PLAYER_WEIGHT_KG"): {
        "strategie": "Flag + conservation",
        "flag": "PLAYER_WEIGHT_IS_MISSING",
        "justification": (
            "La représentation standardisée conserve la valeur manquante."
        ),
    },

    ("rosters", "NUM"): {
        "strategie": "Flag + conservation",
        "flag": "NUM_IS_MISSING",
        "justification": (
            "Un numéro de maillot manquant ne justifie pas "
            "la suppression du joueur."
        ),
    },

    ("rosters", "SCHOOL"): {
        "strategie": "Flag + conservation",
        "flag": "SCHOOL_IS_MISSING",
        "justification": (
            "L'établissement est une information secondaire "
            "et son absence ne rend pas la ligne invalide."
        ),
    },

    ("rosters", "PLAYER_SLUG"): {
        "strategie": "Flag + conservation",
        "flag": "PLAYER_SLUG_IS_MISSING",
        "justification": (
            "Champ technique secondaire. La ligne joueur est conservée."
        ),
    },

    ("rosters", "NICKNAME"): {
        "strategie": "Flag + conservation",
        "flag": "NICKNAME_IS_MISSING",
        "justification": (
            "Le surnom n'est pas nécessaire pour conserver "
            "l'observation joueur."
        ),
    },
}

In [49]:
# Vérification de la couverture des stratégies

strategy_rows = []
uncovered_rows = []


for _, row in missing_before_step6.iterrows():

    key = (
        row["source"],
        row["colonne"]
    )

    strategy = strategy_map.get(
        key
    )

    if strategy is None:

        uncovered_rows.append({
            "source": row["source"],
            "colonne": row["colonne"],
            "manquants": row["manquants"],
            "pourcentage": row["pourcentage"],
        })

        continue

    strategy_rows.append({
        "source": row["source"],
        "colonne": row["colonne"],
        "manquants": row["manquants"],
        "pourcentage": row["pourcentage"],
        "strategie": strategy["strategie"],
        "flag": strategy["flag"],
        "justification": strategy["justification"],
    })


missing_strategy_report = pd.DataFrame(
    strategy_rows
)

uncovered_missing = pd.DataFrame(
    uncovered_rows
)


display(
    missing_strategy_report
)

print(
    "Colonnes manquantes sans stratégie :",
    len(uncovered_missing)
)

display(
    uncovered_missing
)

,source,colonne,manquants,pourcentage,strategie,flag,justification
0,rosters,HOW_ACQUIRED,11280,96.6415,Suppression de colonne,None,"Colonne manquante sur environ 97 % des lignes, absence fortement systématique et faible intérêt pour la prédiction d..."
1,rosters,SUPPLEMENTAL_STATUS,5843,50.0600,Suppression de colonne,None,Plus de 50 % de valeurs manquantes et absence liée à une évolution du schéma de la source roster.
2,kaggle_all_seasons,college,1859,14.4737,Flag + conservation,COLLEGE_IS_MISSING,Environ 14 % de valeurs manquantes. Les lignes restent utiles pour l'analyse et le ML.
3,player_bio,COLLEGE,1821,14.1756,Flag + conservation,COLLEGE_IS_MISSING,Environ 14 % de valeurs manquantes. L'absence n'est pas suffisamment aléatoire pour supprimer les joueurs concernés.
4,player_bio,DRAFT_NUMBER,104,0.8096,Flag + conservation,DRAFT_INFO_IS_MISSING,L'information de draft est conservée sans imputation.
5,player_bio,DRAFT_ROUND,95,0.7395,Flag + conservation,DRAFT_INFO_IS_MISSING,L'information de draft est conservée sans imputation.
6,player_bio,PLAYER_HEIGHT,71,0.5527,Flag + conservation,PLAYER_HEIGHT_IS_MISSING,La ligne joueur reste exploitable malgré une taille absente.
7,player_bio,PLAYER_HEIGHT_INCHES,71,0.5527,Flag + conservation,PLAYER_HEIGHT_IS_MISSING,La ligne joueur reste exploitable malgré une taille absente.
8,player_bio,PLAYER_WEIGHT,71,0.5527,Flag + conservation,PLAYER_WEIGHT_IS_MISSING,Le poids n'est pas imputé avant la séparation train/test.
9,player_bio,PLAYER_HEIGHT_CM,71,0.5527,Flag + conservation,PLAYER_HEIGHT_IS_MISSING,La représentation standardisée conserve la valeur manquante.


Colonnes manquantes sans stratégie : 0


""


In [50]:
# Suppression des colonnes très incomplètes

columns_to_drop = [
    ("rosters", "HOW_ACQUIRED"),
    ("rosters", "SUPPLEMENTAL_STATUS"),
]


for source_name, col in columns_to_drop:

    df = sources_clean[
        source_name
    ]

    if col not in df.columns:
        continue

    missing_count = int(
        df[col].isna().sum()
    )

    total_rows = len(df)

    sources_clean[
        source_name
    ] = df.drop(
        columns=[col]
    ).copy()

    log_step5_once(
        source=source_name,
        colonne=col,
        probleme="Taux de valeurs manquantes critique",
        action="Suppression de la colonne",
        lignes_affectees=total_rows,
        justification=(
            strategy_map[
                (source_name, col)
            ]["justification"]
            + f" Valeurs manquantes : {missing_count}/{total_rows}."
        )
    )


print(
    "HOW_ACQUIRED présente :",
    "HOW_ACQUIRED"
    in sources_clean["rosters"].columns
)

print(
    "SUPPLEMENTAL_STATUS présente :",
    "SUPPLEMENTAL_STATUS"
    in sources_clean["rosters"].columns
)

HOW_ACQUIRED présente : False
SUPPLEMENTAL_STATUS présente : False


In [51]:
# Création des indicateurs de valeurs manquantes

flag_definitions = [
    {
        "source": "player_bio",
        "flag": "COLLEGE_IS_MISSING",
        "colonnes": ["COLLEGE"],
        "justification": (
            "Indique explicitement l'absence d'information sur le college."
        ),
    },

    {
        "source": "player_bio",
        "flag": "DRAFT_INFO_IS_MISSING",
        "colonnes": [
            "DRAFT_YEAR",
            "DRAFT_ROUND",
            "DRAFT_NUMBER",
        ],
        "justification": (
            "Indique qu'au moins une information de draft est absente."
        ),
    },

    {
        "source": "player_bio",
        "flag": "PLAYER_HEIGHT_IS_MISSING",
        "colonnes": [
            "PLAYER_HEIGHT_CM",
        ],
        "justification": (
            "Indique l'absence de taille dans la représentation standardisée."
        ),
    },

    {
        "source": "player_bio",
        "flag": "PLAYER_WEIGHT_IS_MISSING",
        "colonnes": [
            "PLAYER_WEIGHT_KG",
        ],
        "justification": (
            "Indique l'absence de poids dans la représentation standardisée."
        ),
    },

    {
        "source": "rosters",
        "flag": "NUM_IS_MISSING",
        "colonnes": ["NUM"],
        "justification": (
            "Conserve le joueur tout en signalant l'absence du numéro."
        ),
    },

    {
        "source": "rosters",
        "flag": "SCHOOL_IS_MISSING",
        "colonnes": ["SCHOOL"],
        "justification": (
            "Conserve le joueur tout en signalant l'absence d'établissement."
        ),
    },

    {
        "source": "rosters",
        "flag": "PLAYER_SLUG_IS_MISSING",
        "colonnes": ["PLAYER_SLUG"],
        "justification": (
            "Signale l'absence du champ technique sans supprimer la ligne."
        ),
    },

    {
        "source": "rosters",
        "flag": "NICKNAME_IS_MISSING",
        "colonnes": ["NICKNAME"],
        "justification": (
            "Signale l'absence du surnom sans supprimer la ligne."
        ),
    },

    {
        "source": "kaggle_all_seasons",
        "flag": "COLLEGE_IS_MISSING",
        "colonnes": ["college"],
        "justification": (
            "Indique explicitement l'absence d'information sur le college."
        ),
    },
]


flag_report_rows = []


for definition in flag_definitions:

    source_name = definition[
        "source"
    ]

    flag_name = definition[
        "flag"
    ]

    cols = definition[
        "colonnes"
    ]

    df = sources_clean[
        source_name
    ]

    existing_cols = [
        col
        for col in cols
        if col in df.columns
    ]

    if not existing_cols:
        continue

    missing_mask = (
        df[
            existing_cols
        ]
        .isna()
        .any(axis=1)
    )

    df[
        flag_name
    ] = (
        missing_mask
        .astype("Int8")
    )

    affected = int(
        missing_mask.sum()
    )

    flag_report_rows.append({
        "source": source_name,
        "flag": flag_name,
        "colonnes_source": ", ".join(
            existing_cols
        ),
        "lignes_flag_1": affected,
    })

    log_step5_once(
        source=source_name,
        colonne=flag_name,
        probleme="Information manquante conservée",
        action="Création d'un indicateur binaire de valeur manquante",
        lignes_affectees=affected,
        justification=definition[
            "justification"
        ]
    )


missing_flags_report = pd.DataFrame(
    flag_report_rows
)

display(
    missing_flags_report
)

,source,flag,colonnes_source,lignes_flag_1
0,player_bio,COLLEGE_IS_MISSING,COLLEGE,1821
1,player_bio,DRAFT_INFO_IS_MISSING,"DRAFT_YEAR, DRAFT_ROUND, DRAFT_NUMBER",104
2,player_bio,PLAYER_HEIGHT_IS_MISSING,PLAYER_HEIGHT_CM,71
3,player_bio,PLAYER_WEIGHT_IS_MISSING,PLAYER_WEIGHT_KG,71
4,rosters,NUM_IS_MISSING,NUM,30
5,rosters,SCHOOL_IS_MISSING,SCHOOL,6
6,rosters,PLAYER_SLUG_IS_MISSING,PLAYER_SLUG,4
7,rosters,NICKNAME_IS_MISSING,NICKNAME,4
8,kaggle_all_seasons,COLLEGE_IS_MISSING,college,1859


In [52]:
# Vérification de la couverture des valeurs manquantes conservées

coverage_checks = []


for _, row in missing_before_step6.iterrows():

    source_name = row[
        "source"
    ]

    col = row[
        "colonne"
    ]

    key = (
        source_name,
        col
    )

    strategy = strategy_map[
        key
    ]

    if strategy[
        "strategie"
    ] == "Suppression de colonne":

        coverage_checks.append({
            "source": source_name,
            "colonne": col,
            "controle": "Colonne supprimée",
            "erreurs": int(
                col
                in sources_clean[
                    source_name
                ].columns
            ),
        })

        continue


    flag_name = strategy[
        "flag"
    ]

    df = sources_clean[
        source_name
    ]

    if col not in df.columns:

        coverage_checks.append({
            "source": source_name,
            "colonne": col,
            "controle": f"Couverture par {flag_name}",
            "erreurs": 1,
        })

        continue

    missing_mask = df[
        col
    ].isna()

    uncovered = int(
        (
            missing_mask
            & (
                df[
                    flag_name
                ] != 1
            )
        ).sum()
    )

    coverage_checks.append({
        "source": source_name,
        "colonne": col,
        "controle": f"Couverture par {flag_name}",
        "erreurs": uncovered,
    })


missing_coverage = pd.DataFrame(
    coverage_checks
)


display(
    missing_coverage
)


print(
    "Valeurs manquantes non couvertes :",
    int(
        missing_coverage[
            "erreurs"
        ].sum()
    )
)

,source,colonne,controle,erreurs
0,rosters,HOW_ACQUIRED,Colonne supprimée,0
1,rosters,SUPPLEMENTAL_STATUS,Colonne supprimée,0
2,kaggle_all_seasons,college,Couverture par COLLEGE_IS_MISSING,0
3,player_bio,COLLEGE,Couverture par COLLEGE_IS_MISSING,0
4,player_bio,DRAFT_NUMBER,Couverture par DRAFT_INFO_IS_MISSING,0
5,player_bio,DRAFT_ROUND,Couverture par DRAFT_INFO_IS_MISSING,0
6,player_bio,PLAYER_HEIGHT,Couverture par PLAYER_HEIGHT_IS_MISSING,0
7,player_bio,PLAYER_HEIGHT_INCHES,Couverture par PLAYER_HEIGHT_IS_MISSING,0
8,player_bio,PLAYER_WEIGHT,Couverture par PLAYER_WEIGHT_IS_MISSING,0
9,player_bio,PLAYER_HEIGHT_CM,Couverture par PLAYER_HEIGHT_IS_MISSING,0


Valeurs manquantes non couvertes : 0


In [54]:
# Etat des valeurs manquantes après traitement

missing_after_rows = []


for source_name, df in sources_clean.items():

    for col in df.columns:

        count = int(
            df[col].isna().sum()
        )

        if count == 0:
            continue

        missing_after_rows.append({
            "source": source_name,
            "colonne": col,
            "manquants": count,
            "pourcentage": round(
                count / len(df) * 100,
                4
            ),
        })


missing_after_step6 = (
    pd.DataFrame(
        missing_after_rows
    )
    .sort_values(
        ["pourcentage", "source"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


missing_after_step6.to_csv(
    REPORT_DIR / "06_manquants_apres.csv",
    index=False,
    encoding="utf-8"
)


display(
    missing_after_step6
)

,source,colonne,manquants,pourcentage
0,kaggle_all_seasons,college,1859,14.4737
1,player_bio,COLLEGE,1821,14.1756
2,player_bio,DRAFT_NUMBER,104,0.8096
3,player_bio,DRAFT_ROUND,95,0.7395
4,player_bio,PLAYER_HEIGHT,71,0.5527
5,player_bio,PLAYER_HEIGHT_INCHES,71,0.5527
6,player_bio,PLAYER_WEIGHT,71,0.5527
7,player_bio,PLAYER_HEIGHT_CM,71,0.5527
8,player_bio,PLAYER_WEIGHT_KG,71,0.5527
9,rosters,NUM,30,0.2570


In [55]:
# Bilan de l'étape 6

step6_summary_rows = []


for source_name, df in sources_clean.items():

    before_missing = int(
        missing_before_step6.loc[
            missing_before_step6[
                "source"
            ] == source_name,
            "manquants"
        ].sum()
    )

    after_missing = int(
        missing_after_step6.loc[
            missing_after_step6[
                "source"
            ] == source_name,
            "manquants"
        ].sum()
    )

    step6_summary_rows.append({
        "source": source_name,
        "lignes_avant": step6_rows_before[
            source_name
        ],
        "lignes_apres": len(df),
        "variation_lignes": (
            len(df)
            - step6_rows_before[
                source_name
            ]
        ),
        "colonnes_avant": step6_cols_before[
            source_name
        ],
        "colonnes_apres": df.shape[1],
        "variation_colonnes": (
            df.shape[1]
            - step6_cols_before[
                source_name
            ]
        ),
        "manquants_avant": before_missing,
        "manquants_apres": after_missing,
        "variation_manquants": (
            after_missing
            - before_missing
        ),
    })


step6_summary = pd.DataFrame(
    step6_summary_rows
)


step6_summary.to_csv(
    REPORT_DIR / "06_bilan_valeurs_manquantes.csv",
    index=False,
    encoding="utf-8"
)


display(
    step6_summary
)

,source,lignes_avant,lignes_apres,variation_lignes,colonnes_avant,colonnes_apres,variation_colonnes,manquants_avant,manquants_apres,variation_manquants
0,player_bio,12846,12846,0,27,31,4,2375,2375,0
1,player_stats_base,12846,12846,0,69,69,0,0,0,0
2,player_stats_advanced,12846,12846,0,81,81,0,0,0,0
3,team_stats_base,802,802,0,56,56,0,0,0,0
4,team_stats_advanced,802,802,0,48,48,0,0,0,0
5,rosters,11672,11672,0,21,23,2,17167,44,-17123
6,kaggle_all_seasons,12844,12844,0,23,24,1,1859,1859,0


In [56]:
# Validation de l'étape 6

dropped_columns_present = sum(
    int(
        col
        in sources_clean[
            source
        ].columns
    )
    for source, col
    in columns_to_drop
)


flags_with_missing = 0

for definition in flag_definitions:

    source_name = definition[
        "source"
    ]

    flag_name = definition[
        "flag"
    ]

    if flag_name not in sources_clean[
        source_name
    ].columns:
        continue

    flags_with_missing += int(
        sources_clean[
            source_name
        ][flag_name]
        .isna()
        .sum()
    )


rows_changed = sum(
    int(
        len(sources_clean[source])
        != step6_rows_before[source]
    )
    for source in sources_clean
)


step6_validation = pd.DataFrame({
    "controle": [
        "Colonnes à supprimer encore présentes",
        "Valeurs manquantes non couvertes",
        "Valeurs manquantes dans les flags",
        "Sources dont le nombre de lignes a changé",
    ],

    "valeur": [
        dropped_columns_present,
        int(
            missing_coverage[
                "erreurs"
            ].sum()
        ),
        flags_with_missing,
        rows_changed,
    ],
})


display(
    step6_validation
)


assert (
    step6_validation[
        "valeur"
    ] == 0
).all()


print(
    "Tous les contrôles des valeurs manquantes sont valides."
)

,controle,valeur
0,Colonnes à supprimer encore présentes,0
1,Valeurs manquantes non couvertes,0
2,Valeurs manquantes dans les flags,0
3,Sources dont le nombre de lignes a changé,0


Tous les contrôles des valeurs manquantes sont valides.


In [57]:
# Sauvegarde de la documentation

missing_strategy_report.to_csv(
    REPORT_DIR / "06_strategies_valeurs_manquantes.csv",
    index=False,
    encoding="utf-8"
)

missing_flags_report.to_csv(
    REPORT_DIR / "06_flags_valeurs_manquantes.csv",
    index=False,
    encoding="utf-8"
)

cleaning_log = get_cleaning_log()

cleaning_log.to_csv(
    REPORT_DIR / "cleaning_log.csv",
    index=False,
    encoding="utf-8"
)

display(
    cleaning_log
)

,#,source,colonne,probleme,action,lignes_affectees,justification
0,1,player_bio,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
1,2,player_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
2,3,player_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
3,4,team_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
4,5,team_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
5,6,rosters,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
6,7,kaggle_all_seasons,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
7,8,kaggle_all_seasons,Unnamed: 0,Index technique issu de l'export CSV,Suppression de la colonne,12844,La colonne contient uniquement l'index séquentiel des lignes et n'apporte aucune information métier.
8,9,player_bio,PLAYER_WEIGHT,Type de données,Conversion vers Float64,12775,La colonne contient une information numérique et la conversion a été validée sans perte de valeur métier.
9,10,rosters,WEIGHT,Type de données,Conversion vers Float64,11672,La colonne contient une information numérique et la conversion a été validée sans perte de valeur métier.


### Résultat du traitement des valeurs manquantes

Une stratégie explicite a été définie pour chaque colonne contenant des
valeurs manquantes.

Les colonnes `HOW_ACQUIRED` et `SUPPLEMENTAL_STATUS` des rosters ont été
supprimées. Leur taux de valeurs manquantes était respectivement d'environ
97 % et 50 %, et leur absence était fortement liée à la structure historique
de la source.

Les autres observations ont été conservées afin de ne pas supprimer des
joueurs disposant par ailleurs de statistiques exploitables.

Des indicateurs binaires ont été créés pour les informations manquantes
relatives au college, à la draft, aux mesures physiques et à certains champs
des rosters.

Les marqueurs textuels représentant une absence ont été normalisés en
véritables valeurs manquantes.

Aucune moyenne, médiane, mode, interpolation ou autre imputation statistique
n'a été utilisée. L'éventuelle imputation nécessaire au modèle sera effectuée
dans la pipeline ML après la séparation des données.

Aucune ligne n'a été supprimée pendant cette étape.

### Checklist : Valeurs manquantes

- [x] Appliquer la stratégie choisie par colonne
- [x] Créer des flags lorsque nécessaire
- [x] Documenter le traitement appliqué

## 7. Traitement des outliers

Les valeurs extrêmes identifiées pendant la Phase 2 sont réévaluées après
les étapes de nettoyage précédentes.

Une valeur détectée par la méthode IQR n'est pas automatiquement considérée
comme une erreur.

Les règles métier vérifiées pendant le diagnostic n'ont identifié aucune
valeur manifestement impossible. Les valeurs extrêmes observées peuvent
notamment correspondre à des performances NBA atypiques, à de petits
échantillons de matchs ou de minutes, ou à des contextes de saison particuliers.

La stratégie retenue est donc la conservation des valeurs extrêmes légitimes.

Aucune winsorisation, transformation logarithmique ou autre transformation
statistique n'est appliquée à ce stade.

In [58]:
# Etat avant traitement des outliers

step7_rows_before = {
    source: len(df)
    for source, df in sources_clean.items()
}

step7_cols_before = {
    source: df.shape[1]
    for source, df in sources_clean.items()
}

In [59]:
# Colonnes numériques qui ne représentent pas des mesures continues

def is_identifier_or_flag(col):
    name = col.upper()

    if name in {
        "PLAYER_ID",
        "TEAM_ID",
        "TEAMID",
        "LEAGUEID",
    }:
        return True

    if name.endswith("_ID"):
        return True

    if name.endswith("_IS_MISSING"):
        return True

    if name.endswith("_RANK"):
        return True

    if name == "EXP_YEARS":
        return True

    return False

In [60]:
# Détection des valeurs extrêmes par la méthode IQR

outlier_rows = []


for source_name, df in sources_clean.items():

    numeric_cols = df.select_dtypes(
        include=["number"]
    ).columns

    for col in numeric_cols:

        if is_identifier_or_flag(col):
            continue

        values = pd.to_numeric(
            df[col],
            errors="coerce"
        ).dropna()

        if len(values) == 0:
            continue

        if values.nunique() <= 2:
            continue

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)

        iqr = q3 - q1

        if pd.isna(iqr) or iqr == 0:
            continue

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        mask = (
            df[col].notna()
            & (
                (df[col] < lower_bound)
                | (df[col] > upper_bound)
            )
        )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        outlier_rows.append({
            "source": source_name,
            "colonne": col,
            "q1": q1,
            "q3": q3,
            "borne_basse": lower_bound,
            "borne_haute": upper_bound,
            "minimum": values.min(),
            "maximum": values.max(),
            "nombre_outliers": count,
            "pourcentage": round(
                count / len(df) * 100,
                4
            ),
        })


outlier_report = (
    pd.DataFrame(outlier_rows)
    .sort_values(
        "pourcentage",
        ascending=False
    )
    .reset_index(drop=True)
)


outlier_report.to_csv(
    REPORT_DIR / "07_outliers_iqr.csv",
    index=False,
    encoding="utf-8"
)


print(
    "Couples source-colonne avec au moins un outlier IQR :",
    len(outlier_report)
)

display(
    outlier_report.head(20)
)

Couples source-colonne avec au moins un outlier IQR : 130


,source,colonne,q1,q3,borne_basse,borne_haute,minimum,maximum,nombre_outliers,pourcentage
0,team_stats_advanced,MIN,3946.00000,3971.000,3908.500000,4008.500000,2405.0,4016.0,121,15.0873
1,player_stats_base,DD2,0.00000,4.000,-6.000000,10.000000,0.0,71.0,1631,12.6966
2,team_stats_advanced,POSS,7475.00000,7951.000,6761.000000,8665.000000,4325.0,8642.0,66,8.2294
3,player_stats_base,FG_PCT,0.39900,0.481,0.276000,0.604000,0.0,1.0,963,7.4965
4,player_stats_advanced,FG_PCT,0.39900,0.481,0.276000,0.604000,0.0,1.0,963,7.4965
5,player_stats_advanced,E_TOV_PCT,9.00000,13.300,2.550000,19.750000,0.0,100.0,946,7.3642
6,player_stats_advanced,TM_TOV_PCT,9.00000,13.300,2.550000,19.750000,0.0,100.0,945,7.3564
7,player_stats_base,BLK,0.10000,0.500,-0.500000,1.100000,0.0,3.9,923,7.1851
8,player_stats_advanced,EFG_PCT,0.44300,0.527,0.317000,0.653000,0.0,1.5,848,6.6013
9,kaggle_all_seasons,ts_pct,0.48200,0.563,0.360500,0.684500,0.0,1.5,842,6.5556


In [61]:
# Principales valeurs extrêmes détectées

top_outliers = (
    outlier_report[
        [
            "source",
            "colonne",
            "minimum",
            "maximum",
            "borne_basse",
            "borne_haute",
            "nombre_outliers",
            "pourcentage",
        ]
    ]
    .head(30)
)


display(
    top_outliers
)

,source,colonne,minimum,maximum,borne_basse,borne_haute,nombre_outliers,pourcentage
0,team_stats_advanced,MIN,2405.0,4016.0,3908.500000,4008.500000,121,15.0873
1,player_stats_base,DD2,0.0,71.0,-6.000000,10.000000,1631,12.6966
2,team_stats_advanced,POSS,4325.0,8642.0,6761.000000,8665.000000,66,8.2294
3,player_stats_base,FG_PCT,0.0,1.0,0.276000,0.604000,963,7.4965
4,player_stats_advanced,FG_PCT,0.0,1.0,0.276000,0.604000,963,7.4965
5,player_stats_advanced,E_TOV_PCT,0.0,100.0,2.550000,19.750000,946,7.3642
6,player_stats_advanced,TM_TOV_PCT,0.0,100.0,2.550000,19.750000,945,7.3564
7,player_stats_base,BLK,0.0,3.9,-0.500000,1.100000,923,7.1851
8,player_stats_advanced,EFG_PCT,0.0,1.5,0.317000,0.653000,848,6.6013
9,kaggle_all_seasons,ts_pct,0.0,1.5,0.360500,0.684500,842,6.5556


In [62]:
# Vérification des principales incohérences métier

business_outlier_checks = []


def add_business_check(
    source,
    controle,
    count
):
    business_outlier_checks.append({
        "source": source,
        "controle": controle,
        "anomalies": int(count),
    })


for source_name in [
    "player_stats_base",
    "team_stats_base",
]:

    df = sources_clean[source_name]

    for made, attempted in [
        ("FGM", "FGA"),
        ("FG3M", "FG3A"),
        ("FTM", "FTA"),
    ]:

        if {
            made,
            attempted
        }.issubset(df.columns):

            count = (
                df[made].notna()
                & df[attempted].notna()
                & (df[made] > df[attempted])
            ).sum()

            add_business_check(
                source_name,
                f"{made} <= {attempted}",
                count
            )


for source_name in [
    "player_stats_base",
    "player_stats_advanced",
    "team_stats_base",
    "team_stats_advanced",
]:

    df = sources_clean[source_name]

    if {
        "W",
        "L",
        "GP"
    }.issubset(df.columns):

        count = (
            df["W"].notna()
            & df["L"].notna()
            & df["GP"].notna()
            & (
                (
                    df["W"]
                    + df["L"]
                    - df["GP"]
                ).abs() > 0.01
            )
        ).sum()

        add_business_check(
            source_name,
            "W + L = GP",
            count
        )


business_outlier_checks = pd.DataFrame(
    business_outlier_checks
)

display(
    business_outlier_checks
)


print(
    "Anomalies métier détectées :",
    int(
        business_outlier_checks[
            "anomalies"
        ].sum()
    )
)

,source,controle,anomalies
0,player_stats_base,FGM <= FGA,0
1,player_stats_base,FG3M <= FG3A,0
2,player_stats_base,FTM <= FTA,0
3,team_stats_base,FGM <= FGA,0
4,team_stats_base,FG3M <= FG3A,0
5,team_stats_base,FTM <= FTA,0
6,player_stats_base,W + L = GP,0
7,player_stats_advanced,W + L = GP,0
8,team_stats_base,W + L = GP,0
9,team_stats_advanced,W + L = GP,0


Anomalies métier détectées : 0


In [63]:
# Décision de traitement des outliers

outlier_decision = pd.DataFrame([
    {
        "type": "Valeurs impossibles selon les règles métier",
        "decision": "Aucune suppression ou correction",
        "lignes_affectees": 0,
        "justification": (
            "Aucune valeur manifestement impossible n'a été détectée "
            "par les contrôles métier."
        ),
    },
    {
        "type": "Valeurs extrêmes détectées par IQR",
        "decision": "Conservation sans modification",
        "lignes_affectees": 0,
        "justification": (
            "La méthode IQR identifie des observations extrêmes mais "
            "ne permet pas de conclure qu'elles sont erronées. "
            "Dans le contexte NBA, ces valeurs peuvent représenter "
            "des performances légitimes ou de petits échantillons."
        ),
    },
    {
        "type": "Winsorisation",
        "decision": "Non appliquée",
        "lignes_affectees": 0,
        "justification": (
            "Transformation statistique interdite pendant la Phase 3. "
            "Si elle devient nécessaire au modèle, elle devra être "
            "apprise uniquement sur le train dans la pipeline ML."
        ),
    },
    {
        "type": "Transformation log ou racine",
        "decision": "Non appliquée",
        "lignes_affectees": 0,
        "justification": (
            "Aucune transformation statistique globale n'est appliquée "
            "avant la séparation train/test."
        ),
    },
])


outlier_decision.to_csv(
    REPORT_DIR / "07_decisions_outliers.csv",
    index=False,
    encoding="utf-8"
)


display(
    outlier_decision
)

,type,decision,lignes_affectees,justification
0,Valeurs impossibles selon les règles métier,Aucune suppression ou correction,0,Aucune valeur manifestement impossible n'a été détectée par les contrôles métier.
1,Valeurs extrêmes détectées par IQR,Conservation sans modification,0,La méthode IQR identifie des observations extrêmes mais ne permet pas de conclure qu'elles sont erronées. Dans le co...
2,Winsorisation,Non appliquée,0,"Transformation statistique interdite pendant la Phase 3. Si elle devient nécessaire au modèle, elle devra être appri..."
3,Transformation log ou racine,Non appliquée,0,Aucune transformation statistique globale n'est appliquée avant la séparation train/test.


In [64]:
# Journalisation de la décision sur les outliers

already_logged = any(
    row["probleme"] == "Outliers IQR"
    for row in cleaning_log_rows
)


if not already_logged:

    add_cleaning_log(
        source="toutes_sources",
        colonne="colonnes numériques",
        probleme="Outliers IQR",
        action="Conservation sans modification",
        lignes_affectees=0,
        justification=(
            "Les valeurs extrêmes détectées ne violent aucune règle "
            "métier. Elles sont considérées comme des observations "
            "NBA plausibles et sont conservées."
        )
    )

In [65]:
# Bilan structurel de l'étape 7

step7_summary_rows = []


for source_name, df in sources_clean.items():

    step7_summary_rows.append({
        "source": source_name,
        "lignes_avant":
            step7_rows_before[source_name],
        "lignes_apres":
            len(df),
        "variation_lignes":
            len(df)
            - step7_rows_before[source_name],
        "colonnes_avant":
            step7_cols_before[source_name],
        "colonnes_apres":
            df.shape[1],
        "variation_colonnes":
            df.shape[1]
            - step7_cols_before[source_name],
    })


step7_summary = pd.DataFrame(
    step7_summary_rows
)


step7_summary.to_csv(
    REPORT_DIR / "07_bilan_outliers.csv",
    index=False,
    encoding="utf-8"
)


display(
    step7_summary
)

,source,lignes_avant,lignes_apres,variation_lignes,colonnes_avant,colonnes_apres,variation_colonnes
0,player_bio,12846,12846,0,31,31,0
1,player_stats_base,12846,12846,0,69,69,0
2,player_stats_advanced,12846,12846,0,81,81,0
3,team_stats_base,802,802,0,56,56,0
4,team_stats_advanced,802,802,0,48,48,0
5,rosters,11672,11672,0,23,23,0
6,kaggle_all_seasons,12844,12844,0,24,24,0


In [66]:
# Validation de l'étape 7

step7_validation = pd.DataFrame({
    "controle": [
        "Anomalies métier détectées",
        "Sources avec lignes supprimées",
        "Sources avec colonnes modifiées",
    ],

    "valeur": [
        int(
            business_outlier_checks[
                "anomalies"
            ].sum()
        ),

        sum(
            int(
                len(sources_clean[source])
                != step7_rows_before[source]
            )
            for source in sources_clean
        ),

        sum(
            int(
                sources_clean[source].shape[1]
                != step7_cols_before[source]
            )
            for source in sources_clean
        ),
    ],
})


display(
    step7_validation
)


assert (
    step7_validation[
        "valeur"
    ] == 0
).all()


print(
    "Tous les contrôles liés aux outliers sont valides."
)

,controle,valeur
0,Anomalies métier détectées,0
1,Sources avec lignes supprimées,0
2,Sources avec colonnes modifiées,0


Tous les contrôles liés aux outliers sont valides.


In [67]:
# Mise à jour du log

cleaning_log = get_cleaning_log()

cleaning_log.to_csv(
    REPORT_DIR / "cleaning_log.csv",
    index=False,
    encoding="utf-8"
)

display(
    cleaning_log
)

,#,source,colonne,probleme,action,lignes_affectees,justification
0,1,player_bio,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
1,2,player_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
2,3,player_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
3,4,team_stats_base,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
4,5,team_stats_advanced,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
5,6,rosters,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
6,7,kaggle_all_seasons,-,Doublons exacts,Aucune suppression nécessaire,0,Les doublons exacts n'apportent aucune information supplémentaire. La première occurrence est conservée.
7,8,kaggle_all_seasons,Unnamed: 0,Index technique issu de l'export CSV,Suppression de la colonne,12844,La colonne contient uniquement l'index séquentiel des lignes et n'apporte aucune information métier.
8,9,player_bio,PLAYER_WEIGHT,Type de données,Conversion vers Float64,12775,La colonne contient une information numérique et la conversion a été validée sans perte de valeur métier.
9,10,rosters,WEIGHT,Type de données,Conversion vers Float64,11672,La colonne contient une information numérique et la conversion a été validée sans perte de valeur métier.


### Résultat du traitement des outliers

La méthode IQR confirme la présence de valeurs statistiquement extrêmes dans
plusieurs variables numériques.

Ces valeurs ne sont cependant pas considérées comme des erreurs. Les contrôles
métier réalisés pendant le diagnostic et réappliqués pendant le nettoyage ne
détectent aucune valeur manifestement impossible.

Dans le contexte NBA, les valeurs extrêmes peuvent notamment être liées à des
performances atypiques, à de faibles volumes de jeu ou à des contextes de
saison particuliers.

La stratégie retenue est donc la conservation des observations extrêmes sans
modification.

Aucune ligne n'a été supprimée et aucune valeur n'a été corrigée à partir du
seul critère IQR.

Aucune winsorisation ou transformation statistique n'a été appliquée. Si un
traitement statistique des valeurs extrêmes devient nécessaire pour le modèle,
il devra être intégré à la pipeline ML après la séparation des données.

### Checklist : Outliers

- [x] Appliquer le traitement décidé
- [x] Justifier chaque décision

### Bilan

- Outliers statistiques détectés : présents
- Erreurs métier confirmées : **0**
- Lignes supprimées pour outliers : **0**
- Valeurs corrigées pour outliers : **0**
- Winsorisation appliquée : **non**
- Transformation statistique appliquée : **non**

## 8. Validation finale du nettoyage

Cette étape vérifie l'ensemble des transformations réalisées depuis le début
de la Phase 3.

La validation porte sur :

- la comparaison avant/après ;
- les doublons exacts ;
- les valeurs manquantes ;
- les types de données ;
- les formats standardisés ;
- les anomalies métier ;
- les statistiques descriptives ;
- la préservation des données originales ;
- la traçabilité des transformations.

Les sources nettoyées sont ensuite sauvegardées séparément afin d'être
utilisées pendant la Phase 4.

In [79]:
# Validation finale de la Phase 3

# Comparaison avant/après

final_summary_rows = []

for source_name, clean_df in sources_clean.items():

    initial = before_snapshot[
        before_snapshot["source"] == source_name
    ].iloc[0]

    final_summary_rows.append({
        "source": source_name,
        "lignes_avant": int(initial["lignes"]),
        "lignes_apres": len(clean_df),
        "variation_lignes": (
            len(clean_df)
            - int(initial["lignes"])
        ),
        "colonnes_avant": int(initial["colonnes"]),
        "colonnes_apres": clean_df.shape[1],
        "variation_colonnes": (
            clean_df.shape[1]
            - int(initial["colonnes"])
        ),
        "manquants_avant": int(
            initial["cellules_manquantes"]
        ),
        "manquants_apres": count_missing(
            clean_df
        ),
        "doublons_avant": int(
            initial["lignes_doublons_exacts"]
        ),
        "doublons_apres": int(
            clean_df.duplicated(
                keep=False
            ).sum()
        ),
    })


final_comparison = pd.DataFrame(
    final_summary_rows
)

final_comparison[
    "variation_manquants"
] = (
    final_comparison[
        "manquants_apres"
    ]
    - final_comparison[
        "manquants_avant"
    ]
)

final_comparison.to_csv(
    REPORT_DIR / "08_comparaison_avant_apres.csv",
    index=False,
    encoding="utf-8"
)

print("Comparaison avant/après")
display(final_comparison)


# Doublons exacts

total_duplicates = sum(
    int(
        df.duplicated(
            keep=False
        ).sum()
    )
    for df in sources_clean.values()
)


# Valeurs manquantes finales

final_missing_rows = []

for source_name, df in sources_clean.items():

    for col in df.columns:

        missing_count = int(
            df[col].isna().sum()
        )

        if missing_count == 0:
            continue

        final_missing_rows.append({
            "source": source_name,
            "colonne": col,
            "manquants": missing_count,
            "pourcentage": round(
                missing_count
                / len(df)
                * 100,
                4
            ),
        })


final_missing_report = (
    pd.DataFrame(
        final_missing_rows
    )
    .sort_values(
        ["pourcentage", "source"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


final_missing_report.to_csv(
    REPORT_DIR
    / "08_valeurs_manquantes_finales.csv",
    index=False,
    encoding="utf-8"
)


# Vérification que les NaN restants sont attendus

allowed_missing = {
    ("player_bio", "COLLEGE"),
    ("player_bio", "DRAFT_ROUND"),
    ("player_bio", "DRAFT_NUMBER"),
    ("player_bio", "PLAYER_HEIGHT"),
    ("player_bio", "PLAYER_HEIGHT_INCHES"),
    ("player_bio", "PLAYER_WEIGHT"),
    ("player_bio", "PLAYER_HEIGHT_CM"),
    ("player_bio", "PLAYER_WEIGHT_KG"),

    ("rosters", "NUM"),
    ("rosters", "SCHOOL"),
    ("rosters", "NICKNAME"),
    ("rosters", "PLAYER_SLUG"),

    ("kaggle_all_seasons", "college"),
}


unexpected_missing = []

for _, row in final_missing_report.iterrows():

    pair = (
        row["source"],
        row["colonne"]
    )

    if pair not in allowed_missing:

        unexpected_missing.append({
            "source": row["source"],
            "colonne": row["colonne"],
            "manquants": row["manquants"],
        })


unexpected_missing = pd.DataFrame(
    unexpected_missing
)

unexpected_missing_count = len(
    unexpected_missing
)


# Colonnes supprimées

removed_columns_ok = (
    "HOW_ACQUIRED"
    not in sources_clean[
        "rosters"
    ].columns
    and
    "SUPPLEMENTAL_STATUS"
    not in sources_clean[
        "rosters"
    ].columns
)


# Vérification exacte des flags

flag_definitions_final = [
    (
        "player_bio",
        "COLLEGE_IS_MISSING",
        ["COLLEGE"]
    ),
    (
        "player_bio",
        "DRAFT_INFO_IS_MISSING",
        [
            "DRAFT_YEAR",
            "DRAFT_ROUND",
            "DRAFT_NUMBER",
        ]
    ),
    (
        "player_bio",
        "PLAYER_HEIGHT_IS_MISSING",
        ["PLAYER_HEIGHT_CM"]
    ),
    (
        "player_bio",
        "PLAYER_WEIGHT_IS_MISSING",
        ["PLAYER_WEIGHT_KG"]
    ),
    (
        "rosters",
        "NUM_IS_MISSING",
        ["NUM"]
    ),
    (
        "rosters",
        "SCHOOL_IS_MISSING",
        ["SCHOOL"]
    ),
    (
        "rosters",
        "PLAYER_SLUG_IS_MISSING",
        ["PLAYER_SLUG"]
    ),
    (
        "rosters",
        "NICKNAME_IS_MISSING",
        ["NICKNAME"]
    ),
    (
        "kaggle_all_seasons",
        "COLLEGE_IS_MISSING",
        ["college"]
    ),
]


flag_validation_rows = []

for source_name, flag, source_cols in flag_definitions_final:

    df = sources_clean[
        source_name
    ]

    exists = (
        flag in df.columns
    )

    if not exists:

        flag_validation_rows.append({
            "source": source_name,
            "flag": flag,
            "presente": False,
            "binaire": False,
            "correspond_aux_nan": False,
        })

        continue

    expected_flag = (
        df[source_cols]
        .isna()
        .any(axis=1)
        .astype("Int8")
    )

    actual_flag = (
        df[flag]
        .astype("Int8")
    )

    binary_ok = (
        df[flag]
        .dropna()
        .isin([0, 1])
        .all()
    )

    exact_match = (
        actual_flag
        .equals(expected_flag)
    )

    flag_validation_rows.append({
        "source": source_name,
        "flag": flag,
        "presente": True,
        "binaire": bool(binary_ok),
        "correspond_aux_nan": bool(
            exact_match
        ),
    })


flag_validation = pd.DataFrame(
    flag_validation_rows
)

flag_errors = int(
    (
        ~flag_validation[
            "presente"
        ]
    ).sum()
    +
    (
        ~flag_validation[
            "binaire"
        ]
    ).sum()
    +
    (
        ~flag_validation[
            "correspond_aux_nan"
        ]
    ).sum()
)


# Vérification des types importants

expected_types = {
    (
        "player_bio",
        "PLAYER_WEIGHT"
    ): "Float64",

    (
        "rosters",
        "WEIGHT"
    ): "Float64",

    (
        "rosters",
        "AGE"
    ): "Int64",

    (
        "rosters",
        "NUM"
    ): "Int64",

    (
        "rosters",
        "BIRTH_DATE"
    ): "datetime64[ns]",

    (
        "rosters",
        "EXP_YEARS"
    ): "Int64",
}


type_validation_rows = []

for (
    source_name,
    col
), expected in expected_types.items():

    if col not in sources_clean[
        source_name
    ].columns:

        type_validation_rows.append({
            "source": source_name,
            "colonne": col,
            "type_attendu": expected,
            "type_reel": "ABSENT",
            "valide": False,
        })

        continue

    actual = str(
        sources_clean[
            source_name
        ][col].dtype
    )

    type_validation_rows.append({
        "source": source_name,
        "colonne": col,
        "type_attendu": expected,
        "type_reel": actual,
        "valide": actual == expected,
    })


type_validation = pd.DataFrame(
    type_validation_rows
)

type_errors = int(
    (
        ~type_validation[
            "valide"
        ]
    ).sum()
)


# Vérification des espaces externes

space_errors = 0

for source_name, df in sources_clean.items():

    for col in df.columns:

        series = df[col]

        if not (
            pd.api.types.is_object_dtype(
                series
            )
            or pd.api.types.is_string_dtype(
                series
            )
        ):
            continue

        string_values = series.map(
            lambda value:
                value
                if isinstance(value, str)
                else pd.NA
        )

        invalid = (
            string_values.notna()
            & (
                string_values
                != string_values.str.strip()
            )
        )

        space_errors += int(
            invalid.sum()
        )


# Vérification des codes en majuscules

case_validation_rows = []

case_columns = [
    (
        "player_bio",
        "TEAM_ABBREVIATION"
    ),
    (
        "player_stats_base",
        "TEAM_ABBREVIATION"
    ),
    (
        "player_stats_advanced",
        "TEAM_ABBREVIATION"
    ),
    (
        "team_stats_base",
        "TEAM_ABBREVIATION"
    ),
    (
        "team_stats_advanced",
        "TEAM_ABBREVIATION"
    ),
    (
        "kaggle_all_seasons",
        "team_abbreviation"
    ),
]


case_errors = 0

for source_name, col in case_columns:

    df = sources_clean[
        source_name
    ]

    if col not in df.columns:

        case_validation_rows.append({
            "source": source_name,
            "colonne": col,
            "statut": "Non applicable",
            "anomalies": 0,
        })

        continue

    values = (
        df[col]
        .dropna()
        .astype(str)
    )

    invalid = int(
        (
            values
            != values.str.upper()
        ).sum()
    )

    case_errors += invalid

    case_validation_rows.append({
        "source": source_name,
        "colonne": col,
        "statut": "Contrôlé",
        "anomalies": invalid,
    })


case_validation = pd.DataFrame(
    case_validation_rows
)


# Vérification des positions

allowed_positions = {
    "C",
    "C-F",
    "F",
    "F-C",
    "F-G",
    "G",
    "G-F",
}

position_errors = 0

if (
    "POSITION"
    in sources_clean[
        "rosters"
    ].columns
):

    position_errors = int(
        (
            ~sources_clean[
                "rosters"
            ]["POSITION"]
            .dropna()
            .isin(
                allowed_positions
            )
        ).sum()
    )


format_errors = (
    space_errors
    + case_errors
    + position_errors
)


# Vérification des unités standardisées

physical_columns = {
    "player_bio": [
        "PLAYER_HEIGHT_CM",
        "PLAYER_WEIGHT_KG",
    ],

    "rosters": [
        "PLAYER_HEIGHT_CM",
        "PLAYER_WEIGHT_KG",
    ],

    "kaggle_all_seasons": [
        "PLAYER_HEIGHT_CM",
        "PLAYER_WEIGHT_KG",
    ],
}


physical_errors = 0

for source_name, columns in physical_columns.items():

    for col in columns:

        if (
            col
            not in sources_clean[
                source_name
            ].columns
        ):
            physical_errors += 1
            continue

        if not pd.api.types.is_numeric_dtype(
            sources_clean[
                source_name
            ][col]
        ):
            physical_errors += 1


# Vérification de la couverture temporelle

expected_seasons = [
    f"{year}-{str(year + 1)[-2:]}"
    for year in range(
        1996,
        2023
    )
]


season_columns = {
    "player_bio": "SEASON",
    "player_stats_base": "SEASON",
    "player_stats_advanced": "SEASON",
    "team_stats_base": "SEASON",
    "team_stats_advanced": "SEASON",
    "rosters": "SEASON",
    "kaggle_all_seasons": "season",
}


season_validation_rows = []

for source_name, col in season_columns.items():

    actual_seasons = sorted(
        sources_clean[
            source_name
        ][col]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    valid = (
        actual_seasons
        == expected_seasons
    )

    season_validation_rows.append({
        "source": source_name,
        "nombre_saisons":
            len(actual_seasons),
        "saison_min":
            actual_seasons[0]
            if actual_seasons
            else None,
        "saison_max":
            actual_seasons[-1]
            if actual_seasons
            else None,
        "valide":
            valid,
    })


season_validation = pd.DataFrame(
    season_validation_rows
)

season_errors = int(
    (
        ~season_validation[
            "valide"
        ]
    ).sum()
)


# Statistiques descriptives finales

descriptive_rows = []

for source_name, df in sources_clean.items():

    numeric_cols = df.select_dtypes(
        include=["number"]
    ).columns

    if len(numeric_cols) == 0:
        continue

    desc = (
        df[numeric_cols]
        .describe()
        .T
        .reset_index()
        .rename(
            columns={
                "index": "colonne"
            }
        )
    )

    desc.insert(
        0,
        "source",
        source_name
    )

    descriptive_rows.append(
        desc
    )


final_descriptive_stats = pd.concat(
    descriptive_rows,
    ignore_index=True
)

final_descriptive_stats.to_csv(
    REPORT_DIR
    / "08_statistiques_descriptives_finales.csv",
    index=False,
    encoding="utf-8"
)


# Comparaison des statistiques numériques existantes

stats_comparison_rows = []

for source_name in sources_clean:

    raw_df = sources_raw[
        source_name
    ]

    clean_df = sources_clean[
        source_name
    ]

    common_numeric_cols = [
        col
        for col in raw_df.columns
        if (
            col in clean_df.columns
            and pd.api.types.is_numeric_dtype(
                raw_df[col]
            )
            and pd.api.types.is_numeric_dtype(
                clean_df[col]
            )
        )
    ]

    for col in common_numeric_cols:

        before = pd.to_numeric(
            raw_df[col],
            errors="coerce"
        )

        after = pd.to_numeric(
            clean_df[col],
            errors="coerce"
        )

        stats_comparison_rows.append({
            "source": source_name,
            "colonne": col,
            "variation_moyenne": (
                after.mean()
                - before.mean()
            ),
            "variation_mediane": (
                after.median()
                - before.median()
            ),
        })


stats_comparison = pd.DataFrame(
    stats_comparison_rows
)

stats_comparison.to_csv(
    REPORT_DIR
    / "08_statistiques_avant_apres.csv",
    index=False,
    encoding="utf-8"
)


stats_changed = stats_comparison[
    (
        stats_comparison[
            "variation_moyenne"
        ].abs() > 1e-12
    )
    |
    (
        stats_comparison[
            "variation_mediane"
        ].abs() > 1e-12
    )
]

numeric_stats_errors = len(
    stats_changed
)


# Vérification de l'intégrité des sources originales

raw_integrity_rows = []

for source_name, df in sources_raw.items():

    current_fingerprint = int(
        pd.util.hash_pandas_object(
            df,
            index=True
        ).sum()
    )

    initial_fingerprint = (
        raw_fingerprints[
            source_name
        ]
    )

    raw_integrity_rows.append({
        "source": source_name,
        "empreinte_initiale":
            initial_fingerprint,
        "empreinte_finale":
            current_fingerprint,
        "original_inchange": (
            current_fingerprint
            == initial_fingerprint
        ),
    })


raw_integrity = pd.DataFrame(
    raw_integrity_rows
)

raw_errors = int(
    (
        ~raw_integrity[
            "original_inchange"
        ]
    ).sum()
)


# Vérification du cleaning log

cleaning_log = get_cleaning_log()

required_log_columns = {
    "#",
    "source",
    "colonne",
    "probleme",
    "action",
    "lignes_affectees",
    "justification",
}

log_structure_ok = (
    required_log_columns
    .issubset(
        cleaning_log.columns
    )
)

justifications = (
    cleaning_log[
        "justification"
    ]
    .astype("string")
    .str.strip()
)

log_missing_justifications = int(
    (
        justifications.isna()
        | justifications.eq("")
    ).sum()
)

cleaning_log.to_csv(
    REPORT_DIR
    / "cleaning_log.csv",
    index=False,
    encoding="utf-8"
)


# Validation globale

phase3_validation = pd.DataFrame({
    "controle": [
        "Doublons exacts restants",
        "Sources avec nombre de lignes modifié",
        "Colonnes très incomplètes encore présentes",
        "Colonnes avec NaN non prévus",
        "Flags invalides",
        "Types importants invalides",
        "Formats invalides",
        "Colonnes physiques invalides",
        "Couverture temporelle invalide",
        "Statistiques numériques communes modifiées",
        "Sources originales modifiées",
        "Structure du cleaning log invalide",
        "Justifications manquantes dans le log",
    ],

    "valeur": [
        total_duplicates,

        sum(
            int(
                len(
                    sources_clean[
                        source
                    ]
                )
                != len(
                    sources_raw[
                        source
                    ]
                )
            )
            for source in sources_clean
        ),

        int(
            not removed_columns_ok
        ),

        unexpected_missing_count,

        flag_errors,

        type_errors,

        format_errors,

        physical_errors,

        season_errors,

        numeric_stats_errors,

        raw_errors,

        int(
            not log_structure_ok
        ),

        log_missing_justifications,
    ],
})


phase3_validation.to_csv(
    REPORT_DIR
    / "08_validation_phase3.csv",
    index=False,
    encoding="utf-8"
)


print("Validation globale")
display(phase3_validation)

print("Intégrité des données originales")
display(raw_integrity)

print("Validation des flags")
display(flag_validation)

print("Validation temporelle")
display(season_validation)

print("Valeurs manquantes finales")
display(final_missing_report)

print(
    "Colonnes numériques dont les statistiques ont changé :",
    numeric_stats_errors
)

if numeric_stats_errors > 0:
    display(
        stats_changed.head(20)
    )


assert (
    phase3_validation[
        "valeur"
    ] == 0
).all()


print(
    "Tous les contrôles de la Phase 3 sont satisfaits."
)

Comparaison avant/après


,source,lignes_avant,lignes_apres,variation_lignes,colonnes_avant,colonnes_apres,variation_colonnes,manquants_avant,manquants_apres,doublons_avant,doublons_apres,variation_manquants
0,player_bio,12846,12846,0,25,31,6,2233,2375,0,0,142
1,player_stats_base,12846,12846,0,69,69,0,0,0,0,0,0
2,player_stats_advanced,12846,12846,0,81,81,0,0,0,0,0,0
3,team_stats_base,802,802,0,56,56,0,0,0,0,0,0
4,team_stats_advanced,802,802,0,48,48,0,0,0,0,0,0
5,rosters,11672,11672,0,18,23,5,17167,44,0,0,-17123
6,kaggle_all_seasons,12844,12844,0,22,24,2,1859,1859,0,0,0


Validation globale


,controle,valeur
0,Doublons exacts restants,0
1,Sources avec nombre de lignes modifié,0
2,Colonnes très incomplètes encore présentes,0
3,Colonnes avec NaN non prévus,0
4,Flags invalides,0
5,Types importants invalides,0
6,Formats invalides,0
7,Colonnes physiques invalides,0
8,Couverture temporelle invalide,0
9,Statistiques numériques communes modifiées,0


Intégrité des données originales


...

Validation temporelle


,source,nombre_saisons,saison_min,saison_max,valide
0,player_bio,27,1996-97,2022-23,True
1,player_stats_base,27,1996-97,2022-23,True
2,player_stats_advanced,27,1996-97,2022-23,True
3,team_stats_base,27,1996-97,2022-23,True
4,team_stats_advanced,27,1996-97,2022-23,True
5,rosters,27,1996-97,2022-23,True
6,kaggle_all_seasons,27,1996-97,2022-23,True


Valeurs manquantes finales


,source,colonne,manquants,pourcentage
0,kaggle_all_seasons,college,1859,14.4737
1,player_bio,COLLEGE,1821,14.1756
2,player_bio,DRAFT_NUMBER,104,0.8096
3,player_bio,DRAFT_ROUND,95,0.7395
4,player_bio,PLAYER_HEIGHT,71,0.5527
5,player_bio,PLAYER_HEIGHT_INCHES,71,0.5527
6,player_bio,PLAYER_WEIGHT,71,0.5527
7,player_bio,PLAYER_HEIGHT_CM,71,0.5527
8,player_bio,PLAYER_WEIGHT_KG,71,0.5527
9,rosters,NUM,30,0.2570


Colonnes numériques dont les statistiques ont changé : 0
Tous les contrôles de la Phase 3 sont satisfaits.


In [80]:
# Sauvegarde et relecture des sources nettoyées

saved_rows = []


for source_name, df in sources_clean.items():

    output_path = (
        CLEAN_DIR
        / f"{source_name}_clean.parquet"
    )

    df.to_parquet(
        output_path,
        index=False
    )

    reloaded = pd.read_parquet(
        output_path
    )

    same_shape = (
        reloaded.shape
        == df.shape
    )

    same_columns = (
        list(reloaded.columns)
        == list(df.columns)
    )

    saved_rows.append({
        "source": source_name,
        "fichier": str(
            output_path
        ),
        "lignes": len(df),
        "colonnes": df.shape[1],
        "fichier_existe":
            output_path.exists(),
        "dimensions_relecture_ok":
            same_shape,
        "colonnes_relecture_ok":
            same_columns,
    })


clean_files_manifest = pd.DataFrame(
    saved_rows
)


clean_files_manifest.to_csv(
    REPORT_DIR
    / "08_manifest_donnees_nettoyees.csv",
    index=False,
    encoding="utf-8"
)


display(
    clean_files_manifest
)


assert clean_files_manifest[
    "fichier_existe"
].all()

assert clean_files_manifest[
    "dimensions_relecture_ok"
].all()

assert clean_files_manifest[
    "colonnes_relecture_ok"
].all()


phase3_final_summary = pd.DataFrame([
    {
        "element":
            "Sources nettoyées",
        "resultat":
            len(sources_clean),
    },
    {
        "element":
            "Doublons exacts restants",
        "resultat":
            total_duplicates,
    },
    {
        "element":
            "Lignes supprimées",
        "resultat":
            sum(
                len(
                    sources_raw[
                        source
                    ]
                )
                - len(
                    sources_clean[
                        source
                    ]
                )
                for source
                in sources_clean
            ),
    },
    {
        "element":
            "Colonnes très incomplètes supprimées",
        "resultat":
            2,
    },
    {
        "element":
            "Erreurs métier corrigées",
        "resultat":
            0,
    },
    {
        "element":
            "Outliers supprimés",
        "resultat":
            0,
    },
    {
        "element":
            "Sources originales modifiées",
        "resultat":
            raw_errors,
    },
    {
        "element":
            "Fichiers nettoyés sauvegardés",
        "resultat":
            len(
                clean_files_manifest
            ),
    },
])


print("Résumé final")
display(
    phase3_final_summary
)


print(
    "Données nettoyées sauvegardées dans :",
    CLEAN_DIR.resolve()
)

,source,fichier,lignes,colonnes,fichier_existe,dimensions_relecture_ok,colonnes_relecture_ok
0,player_bio,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\player_bio_clean.parquet,12846,31,True,True,True
1,player_stats_base,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\player_stats_base_clean.parquet,12846,69,True,True,True
2,player_stats_advanced,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\player_stats_advanced_clean.p...,12846,81,True,True,True
3,team_stats_base,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\team_stats_base_clean.parquet,802,56,True,True,True
4,team_stats_advanced,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\team_stats_advanced_clean.par...,802,48,True,True,True
5,rosters,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\rosters_clean.parquet,11672,23,True,True,True
6,kaggle_all_seasons,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\kaggle_all_seasons_clean.parquet,12844,24,True,True,True


Résumé final


,element,resultat
0,Sources nettoyées,7
1,Doublons exacts restants,0
2,Lignes supprimées,0
3,Colonnes très incomplètes supprimées,2
4,Erreurs métier corrigées,0
5,Outliers supprimés,0
6,Sources originales modifiées,0
7,Fichiers nettoyés sauvegardés,7


Données nettoyées sauvegardées dans : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean


In [81]:
# Sauvegarde et relecture des sources nettoyées

saved_rows = []


for source_name, df in sources_clean.items():

    output_path = (
        CLEAN_DIR
        / f"{source_name}_clean.parquet"
    )

    df.to_parquet(
        output_path,
        index=False
    )

    reloaded = pd.read_parquet(
        output_path
    )

    same_shape = (
        reloaded.shape
        == df.shape
    )

    same_columns = (
        list(reloaded.columns)
        == list(df.columns)
    )

    saved_rows.append({
        "source": source_name,
        "fichier": str(
            output_path
        ),
        "lignes": len(df),
        "colonnes": df.shape[1],
        "fichier_existe":
            output_path.exists(),
        "dimensions_relecture_ok":
            same_shape,
        "colonnes_relecture_ok":
            same_columns,
    })


clean_files_manifest = pd.DataFrame(
    saved_rows
)


clean_files_manifest.to_csv(
    REPORT_DIR
    / "08_manifest_donnees_nettoyees.csv",
    index=False,
    encoding="utf-8"
)


display(
    clean_files_manifest
)


assert clean_files_manifest[
    "fichier_existe"
].all()

assert clean_files_manifest[
    "dimensions_relecture_ok"
].all()

assert clean_files_manifest[
    "colonnes_relecture_ok"
].all()


phase3_final_summary = pd.DataFrame([
    {
        "element":
            "Sources nettoyées",
        "resultat":
            len(sources_clean),
    },
    {
        "element":
            "Doublons exacts restants",
        "resultat":
            total_duplicates,
    },
    {
        "element":
            "Lignes supprimées",
        "resultat":
            sum(
                len(
                    sources_raw[
                        source
                    ]
                )
                - len(
                    sources_clean[
                        source
                    ]
                )
                for source
                in sources_clean
            ),
    },
    {
        "element":
            "Colonnes très incomplètes supprimées",
        "resultat":
            2,
    },
    {
        "element":
            "Erreurs métier corrigées",
        "resultat":
            0,
    },
    {
        "element":
            "Outliers supprimés",
        "resultat":
            0,
    },
    {
        "element":
            "Sources originales modifiées",
        "resultat":
            raw_errors,
    },
    {
        "element":
            "Fichiers nettoyés sauvegardés",
        "resultat":
            len(
                clean_files_manifest
            ),
    },
])


print("Résumé final")
display(
    phase3_final_summary
)


print(
    "Données nettoyées sauvegardées dans :",
    CLEAN_DIR.resolve()
)

,source,fichier,lignes,colonnes,fichier_existe,dimensions_relecture_ok,colonnes_relecture_ok
0,player_bio,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\player_bio_clean.parquet,12846,31,True,True,True
1,player_stats_base,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\player_stats_base_clean.parquet,12846,69,True,True,True
2,player_stats_advanced,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\player_stats_advanced_clean.p...,12846,81,True,True,True
3,team_stats_base,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\team_stats_base_clean.parquet,802,56,True,True,True
4,team_stats_advanced,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\team_stats_advanced_clean.par...,802,48,True,True,True
5,rosters,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\rosters_clean.parquet,11672,23,True,True,True
6,kaggle_all_seasons,c:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean\kaggle_all_seasons_clean.parquet,12844,24,True,True,True


Résumé final


,element,resultat
0,Sources nettoyées,7
1,Doublons exacts restants,0
2,Lignes supprimées,0
3,Colonnes très incomplètes supprimées,2
4,Erreurs métier corrigées,0
5,Outliers supprimés,0
6,Sources originales modifiées,0
7,Fichiers nettoyés sauvegardés,7


Données nettoyées sauvegardées dans : C:\Users\USER\OneDrive\LiveCampus\master\cours\machine learning\final\Model\data\clean


## Conclusion de la Phase 3

Le nettoyage des sept sources NBA est terminé.

Les étapes prévues par le guide ont été réalisées dans l'ordre : préparation,
doublons, erreurs évidentes, types de données, standardisation, valeurs
manquantes, outliers et validation finale.

Aucun doublon exact n'est présent dans les données finales.

Aucune observation n'a été supprimée pendant le nettoyage. Les observations
contenant des informations secondaires manquantes ont été conservées lorsque
leur suppression n'était pas justifiée.

Les colonnes `HOW_ACQUIRED` et `SUPPLEMENTAL_STATUS` ont été supprimées en
raison de leur taux très élevé de valeurs manquantes.

Les valeurs manquantes restantes disposent d'une stratégie explicite et de
flags lorsque cela est pertinent.

Les types, formats et unités ont été standardisés. Les mappings dont
l'équivalence n'était pas suffisamment certaine n'ont pas été appliqués
automatiquement.

Les valeurs extrêmes détectées par IQR ont été conservées puisqu'aucune règle
métier ne permettait de les considérer comme erronées.

Aucune imputation statistique, winsorisation ou transformation statistique
apprise sur l'ensemble du dataset n'a été utilisée.

Les statistiques descriptives ont été vérifiées après nettoyage.

Les données originales ont été contrôlées et sont restées intactes.

Toutes les transformations sont enregistrées dans le log de nettoyage.

Les sept sources nettoyées ont été sauvegardées au format Parquet dans
`data/clean` et leur relecture a été vérifiée.

### Checklist finale

#### Nettoyage effectué

- [x] Tous les problèmes identifiés au diagnostic sont traités
- [x] Chaque transformation est documentée et justifiée
- [x] Le log de nettoyage est complet

#### Validation

- [x] Tableau comparatif avant/après complété
- [x] Aucun nouveau problème introduit
- [x] Statistiques descriptives vérifiées

#### Qualité

- [x] Les données sont prêtes pour la Phase 4
- [x] Les données originales sont préservées
- [x] La traçabilité est assurée